# 글래드짐 · 자세 → 사람 그림

**하실 일은 세 가지입니다.**

### 1. GPU 켜기 (제일 중요)
위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 고르고 **저장**

### 2. 전부 실행
위 메뉴 **런타임 → 모두 실행** (또는 Ctrl+F9)

경고창이 뜨면 **"무시하고 실행"** 을 누르세요. 구글이 만든 노트가 아니라서 뜨는 정상 안내입니다.

### 3. 기다리기
처음 한 번만 **5분쯤** 걸립니다. 그 뒤로는 한 장에 5초입니다.

다 되면 **맨 아래에 그림 3장**이 나오고 `결과.zip` 이 자동으로 내려받아집니다.

---
사람 생김새를 바꾸시려면 2단계 칸의 **`설명 =`** 줄만 고치시면 됩니다.

In [ ]:
#@title 1단계 · 준비 (3분쯤 걸립니다)
import subprocess, sys
import torch

if not torch.cuda.is_available():
    print('=' * 54)
    print('  GPU가 꺼져 있습니다.')
    print('  위 메뉴 [런타임] -> [런타임 유형 변경] -> T4 GPU -> 저장')
    print('  그 다음 [런타임] -> [모두 실행] 을 다시 눌러 주세요.')
    print('=' * 54)
    raise SystemExit

print('GPU :', torch.cuda.get_device_name(0))
print('파이썬 :', sys.version.split()[0])
print('토치 :', torch.__version__)
print('')
print('필요한 프로그램을 받는 중입니다...')

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'diffusers', 'accelerate', 'safetensors'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('설치가 안 됐습니다. 아래 내용을 그대로 보내 주세요.')
    print('-' * 54)
    print(r.stdout[-2500:])
    print(r.stderr[-2500:])
    raise SystemExit

import importlib
for m in ('diffusers', 'transformers', 'accelerate'):
    try:
        mod = importlib.import_module(m)
        print(m, ':', getattr(mod, '__version__', '?'))
    except Exception as e:
        print(m, '불러오기 실패 :', e)
        raise SystemExit
print('')
print('준비 끝. 아래 칸이 이어서 돌아갑니다.')

In [ ]:
#@title 2단계 · 그림 만들기

# ══════════ 여기만 고치시면 됩니다 ══════════
설명 = 'a fit european man in his early 30s, short brown hair, blue tank top, grey shorts, white sneakers, full body, plain light grey studio background, gym, soft natural light, sharp focus, photorealistic, high detail'

빼고싶은것 = 'cartoon, anime, painting, drawing, blurry, deformed, extra limbs, extra fingers, bad hands, missing limbs, watermark, text, logo, nude, cropped, low quality, worst quality'

같은사람_번호 = 12345    # 숫자를 바꾸면 다른 사람이 나옵니다
# ═════════════════════════════════════════

import base64, io, os, zipfile
import numpy as np
import torch
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from IPython.display import display

POSES = {
  'Squat_1': 'iVBORw0KGgoAAAANSUhEUgAAAwAAAAMACAIAAAAc45fZAAAgAElEQVR4AezBAWEjSbZg0fsgFIWhkANhKNgQ4mFZCC8glCl8CJtYPgRtbVd3jctShmRbJUsZ95xAkiRpMoEkSdJkAkmSpMkEkiRJkwkkSZImE0iSJE0mkCRJmkwgSZI0mUCSJGkygSRJ0mQCSZKkyQSSJEmTCSRJkiYTSJIkTSaQJEmaTCBJkjSZQJIkaTKBJEnSZAJJkqTJBJIkSZMJJEmSJhNIkiRNJpAkSZpMIEmSNJlAkiRpMoEkSdJkAkmSpMkEkiRJkwkkSZImE0iSJE0mkCRJmkwgSZI0mUCSJGkygSRJ0mQCSZKkyQSSJEmTCSRJkiYTSJIkTSaQJEmaTCBJkjSZQJIkaTKBJEnSZAJJkqTJBJIkSZMJJEmSJhNIkiRNJpAkSZpMIEmSNJlAkiRpMoEkSdJkAkmSpMkEkiRJkwkkSZImE0iSJE0mkCRJmkwgSZI0mUCSJGkygSRJ0mQCSZKkyQSSJEmTCSRJkiYTSJIkTSaQJEmaTCBJkjSZQJIkaTKBJEnSZAJJkqTJBJIkSZMJJEmSJhNIkiRNJpAkSZpMIEmSNJlAkiRpMoEkSdJkAkmSpMkEkiRJkwkkSZImE0iSJE0mkCRJmkwgSZI0mUCSJGkygSRJ0mQCSZKkyQSSJEmTCSRJkiYTSJIkTSaQJEmaTCBJkjSZQJIkaTKBJEnSZAJJkqTJBJIkSZMJJEmSJhNI0u48cXghkKQNgSTtxRMHjrwQjeKUTvK7A4cgkLR3gSTtwhMHNnyjM1Q0jgTB0IHvwTOSHlAgSY/viQND3+hsKBobguDIge8cCZ6R9DgCSXp8TxwY+kbnlKIxFASvHPjOhuAZSQ8ikKTH98SBoW90TikaQ0HwjwPfGQqekfQIAkl6cE8cuMA3OkeKxlAQ/OPAd+CFpydeOCV4RtIjCCTp8T1xYOgbnSNF4wJBAN85cOSJF14JnpH0CAJJenxPHBj6RueUojHU6cA3GhueeOGV4BlJdy+QpMf3xIGhb3ROKRpDnf6NxtATL/wleEbSIwgkaReeOLDhG50NRWOo07/RGHrihb8Ez0h6BIEk7cUTB468EECjOKWTBw5s6HTgG42hJ174S/CMpEcQSNLuPHF4IbjYgQNHOh34RuMCT7wEz0h6EIEk6S9FNVqn87tvNIaeeAmekfQ4AknSX4rilG80hp4JJD2UQJL0l6L4XdKA75zxTCDpoQSSpH8UxT+Sxj++s+mZQNKjCSRJ/yiKvySN333nhGcCSQ8okCS9UhSQNDZ8h2co/g0kK5IeUCBJOuHAUPHvZEXSYwokSW8duEgg6TEFkqR/FH9LDpwRSHpYgSQJit8kB84IJD2sQJLmVpyQHDgjkPSwAkmaVTGSHNgUSHpkgSRNqTgvOXBCIOnBBZI0meJSyS8HCCTtRSBJ0yjeIZG0W4EkzaF4h0TSngWStHfF+ySSdi6QpF0r3ieRtH+BJO1U8W6JpCkEkrRHxfskkiYSSNK+FB+RSJpIIEk7UnxEImkugSTtQvERiaQZBZL0+IqPSCRNKpCkB1d8RCJpXoEkPazigxJJUwsk6TEVH5FIEoEkPaDiIxJJ+v8CSXooxQclkvS3QJIeR/ERiST9JpCkB1F8RCJJbwWSdPeKD0ok6YRAku5b8RGJJG0KJOmOFR+RSNJIIEn3qviIRJLOCCTp/hQfkUjSRQJJujPFRySSdKlAku5M8T6JJL1PIEl3o3i3RH85fCeekXSZQJLuQ/E+yZV8P/AcPKjDd47FM5KGAkm6A8X7JJ/2/cCx5+CBHL6zJZ6RtC2QpK9WvEMyVpyQvPH9wJbn4CEcvjMWz0jaEEjSlyreISk+73tj7Dm4f4fvjMUzkjYEkvR1ikslxbV8b4w9B/fv8J2xeEbShkDS9TwdeAl0xtOBlwCKiyTFdX1vjD0Hd+7wnUvEM5JOCSR92tOBYy+BfvN04Ei9BENJcV3fG5d4Du7c4Ttj8YykDYGkz3k6sOUl0N+eDmyol+CUpPhDvjfGnoM7V0X7xlg8I2lDIOkTng6MvQSPaimu5V+NoXoJXkmKP+p7Y+w5uGdV/NC+MRbPSNoQSPqEpwNjL8FjWIo/51+NoXoJ/pIUN/C9MfYc3K0qfmnf2BLPSNoWSPqEpwNjL8EXOzwRL5y0FLfxr8ZQvURS3E7y/cCW5+BuVfFG+8axeEbSUCDpo54OXOIl+AKHJ47FC0txY/9qXOKl82clb3w/cOw5uFtVnJTJD4fvxDOSLhNI+oSnA2MvwRc4PLHl3//h9v7VGHvp/BHJQCt++k/jfzo/9eQ+VbElE0nvFEj6hKcDYy/BrR2eGPv3f7ixfzXGXjpXkLxLK4715A5VsSUTSe8XSPqEpwNjL8GtHZ4Y+/d/uLF/NcZegt8UJyTX1YpjPbk3VQxkIun9Akmf83Rgy0vwBQ5PjP37P9zYmjwd2PISfIlWnNST+1HFQCaSPiSQ9Em1PLWVIy/BFzg8cYl//4c/bU3eeDpw7CX4Kq04qSd3ooqxTCR9SCDpk2rhH09tfekLufKFDk+M/fs//CFrsq2Kn/J/Dvwn+CX5Gq04qSd3ooqBTCR9VCDpM2rhWK58ocMTY//+D1exJu9RxS/JK8nXaMVJPbkHVYxlIumjAkmfUQvHcuWrVKP9L2Pxws1V8VrySvI1WrGlJ1+rirFMJH1CIOkzauFYrnyJavzU/pct8cJXqOK15JXka7RiS0++UBVjmUj6nEDSh9XCSblye9V4rf0vx+KFL1LFa8kryddoxZaefJUqzspE0ucEkj6sFk7KlRurxpb2v8QLX62K15JXkq/Rii09+RJVnJWJpE8LJH1YLZyUKzdTjYHs3IEq3kh+l3yBVmzpye1VcVYmkq4hkPQxtbAlV26jGgPZuQ9VvJH8LvkCrdjSk9ur4qxMJF1DIOljamFLrtxANcaycx+qeCP5XfIFWjHQk1uq4qxMJF1JIOljamFLrvxp1RjLzn2o4ljyu+QLtGKgJzdTxVmZSLqeQNLH1MKWXPmjqjGWnbtRxbHkd8kXaMVAT26mirMykXQ9gaSPqYUtufLnVOOs7NyNKo4lR5Jba8VAT26jirMykXRVgaQPqIWBXPlDqnFWdu5JFW8kpyS31oqBntxAFZfIRNJVBZI+oBYGcuVPqMZZ2bknVRxLTklurRUDPfnTqrhEJpKuLZD0AbUwkCtXV41LZOeeVHEsOSW5tVaM9eSPquKsTCT9AYGkD6iFgVy5rmpcIjt3popjySnJrbVirCd/ThWXyETSHxBIeq9aGMuVK6rGJbJzf6o4lpyS3Forxnryh1RxiUwk/RmBpPeqhbFcuZZqXCI796eKk5INyU21Yqwnf0IVl8hE0h8TSHqvWhjLlauoxoWyc3+qOCnZkNxUK8Z68idUcYlMJP0xgaT3qoWBXLmKalwoO3epipOSDclNtWKsJ9dVxYUykfQnBZLeqxYGcuXzqnGh7NyrKk5KNiQ31YqxnlxXFZfIRNIfFkh6l1oYy5VPqsaFsnPHqjiWCcVpyU21YqwnV1TFJTKR9OcFkt6lFsZy5TOqcaHs3LEqTsqE4rTk1lox0JNrqeJCmUj68wJJ71ILY7nyYdW4UHbuWxUnZUJxWnJrrRjryedVcaFMJN1EIOldamEsVz6mGpfLzn2r4qRMKDYlN9WKsZ58XhUXykTSTQSS3qUWxnLlY6pxoezcvSpOyoRiU3JTrRjrySdVcaFMJN1KIOldamEsV96rGpfLzt2rYksmFJuSm2rFWE8+o4oLZSLphgJJ71ILY7nyLtW4XHYeQRVbMqHYlNxUK8Z68hlVXCITSbcVSLpcLYzlyrtU43LZeRBVbMmEYlNyU60Y68mHVXGhTCTdViDpcrUwlivvUo0LZedxVLElk/+vOC25qVaM9eRjqrhQJpJuLpB0uVoYy5XLVeNC2XkoVZyUyd+K05KbasVYTz6gigtlIukrBJIuVwtjuXKhalwoOw+lii2Z/K04LbmpVoz15AOquFAmkr5CIOlytTCWK5eoxuWy81Cq2JLJ34pNye20Yqwn71XFhTKR9EUCSZerhbFcOasal8vOo6liSyZ/KzYlt9OKsZ68VxWXyETS1wkkXa4WxnJlrBqXy84DqmJLJn8rNiU31YqBnrxLFZfIRNKXCiRdqBbGcuWsalwoOw+oioFM/lZsSm6qFWM9uVAVF8pE0pcKJF2oFsZyZawaF8rOY6piIJO/FZuSm2rFWE8uUcUlMpF0BwJJF6qFsVwZqMaFsvOwqtiSyX8Vm5KbasVYTy5RxSUykXQHAkkXqoWxXNlSjctl52FVsSWT/yo2JTfVirGenFXFJTKRdB8CSReqhbFcOakal8vOI6tiSyb/VWxKbqoVYz0Zq+ISmUi6G4GkC9XCWK4cq8blsvPgqtiSyX8VI8nttGKsJ2NVnJWJpHsSSLpQLYzlyhvVuFB2Hl8VA5n8VzGS3E4rxnoyUMUlMpF0TwJJF6qFgVx5oxqXy87jq2Igk/8qRpLbacVYT7ZUcYlMJN2ZQNKFamEgV96oxoWyswtVDGTyX8VIclOtGOjJlirOykTS/QkkXaIWxnLltWpcKDt7UcWWTH5TjCQ31YqBnpxUxVmZSLpLgaRL1MJYrvxSjQtlZy+qGMjkN8VIclOtGOjJSVWclYmkuxRIukQtjOXKT9W4UHZ2pIqBTH5TjCQ31YqBnhyr4qxMJN2rQNIlamEgV36qxoWysy9VDGTym2IkualWDPTkjSrOykTSHQskXaIWBnLlh2pcKDu7U8VAJm8Vm5KbasVAT96o4qxMJN2xQNIlamEgV36oxiWys0dVDGTyVjGS3FQrtvTktSrOykTSfQskXaIWBnKlGpfIzh5VMZDJCcVIclOt2NKTX6o4KxNJdy+QdIla2JIr1bhEdnaqioFMTihGkptqxZae/FLFWCaSHkEg6RK1sGnhEtnZryoGMjmhGEluqhVbevJTFWOZSHoQgaRL1MKmhbOys2tVDGRyQjGS3FQrtvTkhyrGMpH0OAJJl6iFTQtj2dm1KsYyOaEYSW6qFVt68kMVA5lIeiiBpEvUwqaFgezsXRVjmZxQjCS31oqTelLFWCaSHkog6RK1cNrCQHYmUMVYJicUI8mtteKknlQxkImkRxNIukQtnLawJTtzqGIgk9OKM5KbasVJCyOZSHpAgaRL1MJpCydlZxpVDGRyWnFGclOtOGlhUyaSHlMg6RK1cMLCSdmZRhVjmZxWnJHcWiveWNiUiaSHFUi6RC2csHAsOzOpYiyT04ozkltrxRsLp2Ui6ZEFki5RCycsvJad+VQxlslpxRnJrbXil2r0zkmZSHpwgaRL1MIPbaUv/G3htexMqYqxTE4rzkhurRXVONY7v2Qi6fEFks46cEJv/JKdWVUxkMmm4ozk1g4HtvTOD5lI2oVA0tiBTb3xQ3ZmVcVYJiPFSHJThwNjEUjai0DSwIEzgplVMZbJSDGS3NThwFgEkvYikDRw4IxgZlWMZTJSjCQ3dTgwFoGkvQgkDRw4I5hZFWOZjBRnJDdyOHCJCCTtQiBpy4GLBNOqYiyTkeKM5HYOB8YikLQXgaSBA2cEM6tiLJOR4ozkdg4HxiKQtBeBpIEDZwQzq2IgkzOKM5LbORwYi0DSXgSSBg6cEUyrirFMzijOSG7qcGBLBJJ2JJA0dmBTMLMqxjI5ozgjubXDgWMRSNqXQNJZB04IJlfFWCZnFGckX+ZwIAJJOxVIutwBAv1UxVgmZxRnJJL0JwSS9CFVDGRyXnFGIkl/QiBJ71fFWCbnFWckkvQnBJL0flWMZXKR4oxEkq4ukKT3q2Isk4sUZySSdHWBJL1fFWOZXKQ4I5Gkqwsk6f2qGMvkIsUZiSRdXSBJ71TFWZlcpDgjkaSrCyTpnaoYy+RSxRmJJF1dIEnvVMVYJpcqzkgk6eoCSXqnKsYyuVRxRiJJVxdI0jtVMZbJpYozEkm6ukCS3qOKszK5VHFGIklXF0jSe1Qxlsk7FGckknR1gSS9RxVjmbxDcV4iSdcVSNJ7VDGWyfsUZySSdF2BJL1HFWOZvE9xRiJJ1xVI0ntUMZbJ+xRnJJJ0XYEkXayKszJ5n+KMRJKuK5Cki1VxVibvU5yRSNJ1BZJ0sSrGMnm34oxEkq4rkKSLVTGWybsVZySSdF2BJF2sirFM3q04I5Gk6wok6TJVnJXJuxVnJJJ0XYEkXaaKszJ5t+KMRJKuK5Cky1QxlslHFGckknRdgSRdpoqxTD6iOC+RpCsKJOkyVYxl8kHFGYkkXVEgSZepYiyTDyrOSCTpigJJukwVY5l8UHFGIklXFEjSBao4K5MPKs5IJOmKAkm6QBVjmXxccUYiSVcUSNIFqhjL5OOKMxJJuqJAki5QxVgmH1eckUjSFQWSdIEqxjL5uOKMRJKuKJCkc6o4K5OPK85IJOmKAkk6p4qxTD6lOCORpCsKJOmcKsYy+ZTijESSriiQpHOqGMvks4ozEkm6lkCSzqliLJPPKs5IJOlaAkk6p4qxTD6rOCORpGsJJOmcKgYyuYLijESSriWQpKEqxjK5guKMRJKuJZCkoSrGMrmC4oxEkq4lkKShKsYyuYLijESSriWQpKEqxjK5guKMRJKuJZCkoSoGMrmO4oxEkq4lkKRtVYxlch3FGYkkXUsgSduqGMvkaoqRRJKuJZCkbVWMZXI1xUgiSdcSSNK2KsYyuZpiJJGkawkkaVsVY5lcTXFGIklXEUjStirGMrma4oxEkq4ikKRtVQxkck3FGYkkXUUgSRuqGMvkmoozEkm6ikCSNlQxlsk1FWckknQVgSRtqGIsk2sqzkgk6SoCSdpQxVgm11SckUjSVQSStKGKgUyurxhJJOkqAknaUMVAJtdXjCSSdBWBJJ1SxVgm11eMJJJ0FYEknVLFWCbXV4wkknQVgSSdUsVYJtdXjCSSdBWBJJ1SxUAmf0QxkkjSVQSSdEoVA5n8EcVIIklXEUjSKVUMZPJHFCOJJF1FIEmnVDGQyZ9SbEok6SoCSTpSxUAmf1AxkkjS5wWSdKSKgUz+oGIkkaTPCyTpSBUDmfxBxUgiSZ8XSNKRKgYy+YOKkUSSPi+QpCNVDGTyBxUjiSR9XiBJR6rYksmfVYwkkvR5gST9roqBTP6sYiSRpM8LJOl3VQxk8mcVI4kkfV4gSb+rYiCTP67YlEjS5wWS9LsqtmRyC8WmRJI+L5Ck31WxJZNbKDYlkvR5gST9rootmdxCsSmRpM8LJOl3VWzJ5BaKTYkkfV4gSa9UMZDJLRSbEkn6vECSXqliIJNbKDYlkvR5gSS9UsWWTG6k2JRI0ucFkvRKFVsyuZ3itESSPi+QpFeq2JLJ7RSnJZL0eYEkvVLFlkxupzgtkaTPCyTplSpOyuSmik2JJH1SIEn/qGJLJjdVbEok6ZMCSfpHFVsyualiUyJJnxRI0j+q2JLJTRWbEkn6pECS/lHFlkxuqtiUSNInBZL0jypOyuQLFKclkvRJgST9o4qTMvkCxWmJJH1SIEn/qOKkTL5AcVoiSZ8USNI/qjgpky9QnJZI0icFkvSPKo5l8jWK0xJJ+qRAkv5SxUmZfI3itESSPimQpL9UcVImX6M4LZGkTwok6S9VnJTJlylOSCTpkwJJ+ksVJ2XyZYoTEkn6pECS/lLFsUy+UnFCIkmfFEjSX6o4lslXKk5IJOmTAkn6SxXHMvlKxQmJJH1SIEl/qeJYJl+pOCGRpE8KJOkvVbyRydcr3kok6ZMCSYIqjmXy9Yq3Ekn6pECSoIpjmXy94q1Ekj4pkCSo4lgmX694K5GkTwokCap4I5O7ULyVSNInBZIEVbyRyb0ofpNI0icFkgRVvJHJvSh+k0jSJwWSBFW8lskdKX6TSNInBZIEVbyWyR0pfpNI0icFkgRVvJbJHSl+k0jSJwWSBFW8lskdKV773ngOJOkzAkmCKn7J5O4U3xvHngNJ+oBAkqCK9j/0//BDJvfm+4Etz4EkvVcgaW6HJ47FC/fj+4Gx50CS3iWQNLHDE1vihTvx/cDYcyBJ7xJImtXhibF44R58PzD2HEjSuwSSZnV4Yixe+HLfD1ziOZCkywWSZnV4YixeuAffD4w9B5L0LoGkKR2euES88OW+Hxh7DiTpXQJJszo8MRYv3IPvB8aeA0l6l0DSrA5PjMULd+L7gS3PgSS9VyBpVocnxuKF+/H9wLHnQJI+IJA0scMTW+KF+/T9wP/p/LImkvRegaS5HZ44Fi/craV4bU0k6b0CSdOrxg/tf+nf+CE792wpXlsTSXqvQNL0qvFadu7ZUry2JpL0XoGk6VXjtezcs6V4Y00k6V0CSdOrxhvZuVtL8caaSNK7BJKmV403snO3luKNNZGkdwkkza0ax7Jzt5bijTWRpHcJJM2tGseyc7eW4tiaSNLlAklzq8ax7NytpTi2JpJ0uUDS3KpxLDt3aymOrYkkXS6QNLdqHMvO3VqKY2siSZcLJM2tGseyc7eW4qQ1kaQLBZLmVo1j2blbS3HSmkjShQJJc6vGsezcraU4aU0k6UKBpLlV41h27tZSnLQmknShQNLcqnEsO3drKU5aE0m6UCBpYtU4KTt3aym2rIkkXSKQNLFqnJSde7YUJ62JJF0ikDSxamzJzt1aipPWRJIuEUiaWDW2ZOduLcVJayJJlwgkTawaW7Jzt5bipDWRpEsEkiZWjS3ZuVtLsWVNJOmsQNLEqrElO3drKbasiSSdFUiaWDW2ZOduLcWWNZGkswJJE6vGluzcraXYsiaSdFYgaWLV2JKdu7UUW9ZEks4KJE2sGluyc7eWYmBNJGkskDSxamzJzt1aioE1kaSxQNLEqrElO3drKQbWRJLGAkkTq8aW7NyzpdiyJpI0FkiaVTUGsnPPlmLLmkjSWCBpVtUYyM49W4otayJJY4GkWVVjIDv3bCkG1kSSBgJJs6rGQHbu2VIMrIkkDQSSZlWNgezcs6UYWBNJGggkzaoaA9m5Z0sxsCaSNBBImlU1BrJzz5ZiYE0kaSCQNKtqDGTnni3F2JpI0pZA0qyqMZCdO7cUA2siSVsCSbOqxkB27txSDKyJJG0JJM2qGgPZuXNLMbAmkrQlkDSragxk584txcCaSNKWQNKsqjGQnTu3FANrIklbAkmzqsZAdu7cUoytiSSdFEiaVTUGsnPnlmJsTSTppEDSrKoxkJ07txRjayJJJwWSZlWNgezcuaUYWxNJOimQNKtqDGTnzi3F2JpI0kmBpFlVYyA7928pBtZEkk4KJM2qGgPZuX9LMbAmknRSIGlW1RjIzv1birE1kaRjgaRZVWMgO/dvKcbWRJKOBZKmVI2x7Ny/pRhbE0k6FkiaUjXGsnP/lmJsTSTpWCBpStUYy879W4qxNZGkY4GkKVVjLDv3bynG1kSSjgWSplSNsew8hKUYWBNJOhZImlI1xrLzEJZibE0k6Y1A0pSqMZadh7AUY2siSW8EkqZUjbHsPISlGFsTSXojkDSlaoxl5yEsxdiaSNIbgaQpVWMsOw9hKcbWRJLeCCRNqRpj2XkIS3HWmkjSa4GkKVVjLDuPYinG1kSSXgskTakaY9l5FEsxtiaS9FogaUrVGMvOo1iKsTWRpNcCSVOqxlh2HsVSjK2JJL0WSJpSNcay8yiWYmxNJOm1QNKUqjGWnUexFGNrIkmvBZKmVI2x7DyKpThrTSTpl0DSlKoxlp1HsRRnrYkk/RJImlI1xrLzQJZibE0k6ZdA0pSqMZadB7IUY2siSb8EkqZUjbHsPJClGFsTSfolkDSlaoxl54EsxdiaSNIvgaQpVWMsOw9kKcbWRJJ+CSRNqRpj2XkgS3HWmkjST4GkKVVjLDsPZCnOWhNJ+imQNKVqjGXnsSzF2JpI0k+BpClVYyw7j2UpxtZEkn4KJE2pGmPZeSxLMbYmkvRTIGlK1RjLzmNZirE1kaSfAklTqsZYdh7LUoytiST9FEiaUjXGsvNYlmJsTSTpp0DSlKoxlp3HshRnrYkk/RBImlI1xrLzcJZibE0k6YdA0pSqMZadh7MUY2siST8EkqZUjbHsPJylGFsTSfohkDSlaoxl5+EsxdiaSNIPgaQpVWMgO49oKcbWRJJ+CCRNqRoD2XlQSzGwJpL0QyBpStUYyM6DWoqxNZGkQNKUqjGQnQe1FGNrIkmBpClVYyA7D2opxtZEkgJJU6rGQHYe1FKMrYkkBZKmVI2B7DyopRhbE0kKJE2pGgPZeVBLMbYmkhRImlI1BrLzuJZiYE0kKZA0pWoMZOdxLcXAmkhSIGlK1RjIzuNairE1kTS5QNKUqjGQnce1FGNrImlygaQpVWMgO49rKcbWRNLkAklTqsZAdh7XUoytiaTJBZKmVI2B7DyupRhbE0mTCyRNqRoD2XloSzGwJpImF0iaUjUGsvPQlmJgTSRNLpA0pWoMZOehLcXAmkiaXCBpStUYyM5DW4qBNZE0uUDSlKoxkJ2HthQDayJpcoGkKVVjIDsPbSnG1kTSzAJJU6rGQHYe3VIMrImkmQWSplSNLdnZgaUYWBNJMwskTakaW7KzA0sxsCaSZhZImlI1tmRnB5ZiYE0kzSyQNKVqbMnODizFwJpImlkgaUrV2JKdfViKLWsiaWaBpClVY0t29mEptqyJpJkFkqZUjS3Z2Yel2LImkmYWSJpSNbZkZx+WYmBNJE0rkDSlamzJzj4sxcCaSJpWIGlK1diSnX1YioE1kTStQNKUqrElO7uxFFvWRNK0AklTqsaW7OzGUmxZE0nTCiTNqhonZWc3lmLLmkiaViBpVtU4KTu7sRRb1kTStAJJs6rGsezsyVJsWRNJ0wokzaoax7KzM0tx0ppImlYgaVbVOJadnVmKk9ZE0rQCSbOqxrHs7MxSnLQmkqYVSJpVNY5lZ2eW4qQ1kTStQNKsqnEsOzuzFFvWRNKcAkmzqsax7OzPUpy0JpLmFEiaVTWOZWd/lgP+W3oAAAxoSURBVOKkNZE0p0DSrKpxLDv7sxQnrYmkOQWSZlWNY9nZn6U4aU0kzSmQNKtqvJGdXVqKk9ZE0pwCSbOqxhvZ2aulOLYmkuYUSJpYNV7Lzl4txbE1kTSnQNLEqvFadvZqKY6tiaQ5BZImVo3XsrNXS3FsTSTNKZA0sWr80Dq98UN2dmwp3lgTSXMKJM3qwAnBbi3FG2siaU6BpCkd2BTs01K8sSaS5hRIms+BM4IdWoo31kTSnAJJ8zlwRrBPS/HamkiaUyBpPgfOCPZpKV5bE0lzCiRN5sBFgh1aitfWRNKcAknzOXBGsE9L8dqaSJpTIGk+B84Idmspfvm/jQgkTSiQNJ8DZwS7tRT/t3EsAknzCCRN6cCmYM8OB7ZEIGkSgaRZHTgh2LPDgbEIJM0gkDSxWvihrfSFH3Jl3w4HxiKQNINA0sRq4bVc2bfDgbEIJM0gkDSxWngtV3bscOASEUjavUDSrGrhjVzZt8OBsQgkzSCQNKtaeCNX9u1wYCwCSTMIJM2qFt7IlX07HBiLQNIMAkmzqoU3cmX3Dge2RCBpEoGkKdXCsVyZweHAsQgkzSOQNKVaOJYr82hFNbLzU08kzSOQNKVaOJYr82jFaz2RNI9A0pRq4ViuzKMVr/VE0jwCSVOqhWO5Mo9WvNETSZMIJM2nFk7KlXm04o2eSJpEIGk+tXBSrsyjFW/0RNIkAknzqYWTcmUerXijJ5ImEUiaTy2clCvzaMUbPZE0iUDSfGrhpFyZRyuO9UTSDAJJ86mFk3JlHq041hNJMwgkTaYWtuTKPFpxrCeSZhBImkwtbMmVebTiWE8kzSCQNJla2JIr82jFsZ5ImkEgaTK1sCVX5tGKYz2RNINA0mRqYUuuzKMVJ/VE0u4FkiZTC1tyZR6tOKknknYvkDSTWhjIlXm04qSeSNq9QNJMamEgV+bRipN6Imn3AkkzqYUtuTKVVpzUE0m7F0iaSS1syZWptOKknkjavUDSTGphS65MpRUn9UTS7gWSZlILW3JlKq3Y0hNJ+xZImkktbMmVqbRiS08k7VsgaRq1MJArU2nFlp5I2rdA0jRqYSBXptKKLT2RtG+BpGnUwkCuTKUVW3oiad8CSdOohYFcmUortvRE0r4FkqZRCwO5MpVWbOmJpH0LJE2jFgZyZSqtGOiJpB0LJE2jFgZyZSqtGOiJpB0LJM2hFsZyZSqtGOiJpB0LJM2hFsZyZSqtGOiJpB0LJM2hFgZyZTatGOiJpB0LJM2hFgZyZTatGOiJpB0LJM2hFgZyZUKt2NITSTsWSJpDLQzkyoRasaUnknYskDSHWhjIlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjLlQm1YqAnkvYqkDSBWhjIlTm1YqwnknYpkDSBWhjIlTm1YqwnknYpkLR3tTCWK3NqxVhPJO1SIGnvamEsV+bUirGeSNqlQNLe1cJYrsypFWM9kbRLgaS9q4WxXJlTK8Z6ImmXAkl7VwtjuTKnVoz1RNIuBZL2rhbGcmVarRjoiaRdCiTtXS2M5cq0WjHQE0m7FEjau1oYy5VptWKsJ5L2J5C0a7Uwlisza8VYTyTtTyBp12phLFdm1oqxnkjan0DSrtXCWK7MrBVjPZG0P4GkXauFsVyZWSvGeiJpfwJJu1YLY7kys1aM9UTS/gSSdq0WxnJlZq0Y64mk/Qkk7VotjOXKzFox1hNJ+xNI2rVaGMuVmbVirCeS9ieQtGu1MJYrM2vFWE8k7U8gaddqYSxXZtaKs3oiaWcCSbtWC2O5MrlWjPVE0s4EkvarFsZyRa0Y64mknQkk7VctjOWKWjHWE0k7E0jar1oYyxW1YqwnknYmkLRftTCWK2rFWE8k7Uwgab9qYSxX1IqxnkjamUDSftXCWK6oFWM9kbQzgaT9qoWxXFErxnoiaWcCSftVC2O5olaM9UTSzgSS9qsWBnJFP7RirCeSdiaQtF+1MJAr+qEVYz2RtDOBpJ2qhbFc0Q+tOKsnkvYkkLRTtTCWK/qpFWM9kbQngaSdqoWxXNFPrRjriaQ9CSTtVC2M5Yp+asVYTyTtSSBpp2phLFf0UyvGeiJpTwJJO1ULY7min1ox1hNJexJI2qlaGMsV/dSKsZ5I2pNA0k7VwkCu6JdWjPVE0p4EknaqFgZyRb+0YqwnkvYkkLRTtTCQK/qlFWM9kfT/2oO74lh6MwyAzwshWAaLDWGESYJgYxksgbC5+KpSzv7o7Mmdpe5eSQVYVD8y0a7wX2fP3GgBVlIBFtWPTLQr/HT2TIwWYCUVYEX9yFy7wk9nz8RoAVZSAVbUj8y1K/x09syNFmAZFWBF/chEu8Kds2dutADLqAAr6kcm2hXunD1zowVYRgVYUT8y0a5w5+yZGy3AMirAivqRiXaFO2fP3GgBllEBVtSPTLQr3Dl75kYLsIwKsKJ+ZKJd4c7ZMzdagGVUgBX1IxPtCo/OnonRAiyjAqyoH3mlXeGps2ditADLqAAr6kdeaVd46uyZGC3AMirAivqRV9oVnjp7JkYLsIwKsKJ+5JV2hafOnonRAiyjAqyoH3mlXeGps2ditADLqADL6Ucm2hWeOnsmRguwjAqwnH5kol3hqbNnYrQAy6gAy+lHXmlXmDh7XhktwDIqwHL6kVfaFSbOnldGC7CMCrCcfuSVdoWJs2ditABrqADL6UdeaVeYOHsmRguwhgqwnH7klXaFibNnYrQAa6gAy+lHXmlXmDh7JkYLsIYKsJx+5Kl2hbmzZ2K0AGuoAMvpR55qV5g7eyZGC7CGCrCcfuSpdoU/OnteGS3AGirAcvqRp9oV/ujseWW0AGuoAMvpR55qV/ijs+eV0QKsoQIspx95ql3hj86eV0YLsIYKsJx+5FG7wjvOnldGC7CGCrCcfuRRu8I7zp5XRguwhgqwnH7kUbvCO86eV0YLsIYKsJx+5FG7wpvOnqdGC7CGCrCcfuRRu8Kbzp6nRguwhgqwnH7kTrvC+86ep0YLsIYKsJx+5E67wvvOnqdGC7CGCrCcfuROu8L7zp6nRguwhgqwnH7kTrvC+86ep0YLsIYKsJx+5Kd2hb9y9jw1WoA1VIDl9CM/tSv8rbPn0WgB1lABltOP/OO8Psbx3a7wt86eR6MFWEMFWM4tH3lQ+Q5vO3sejRZgDRVgLbd85IXKd3jP2fNotABrqAALueUjU5Xv8Iaz59FoAdZQARZyy0emKt/hDWfPo9ECrKECLOSWj0xVvsN7zp47owVYQwVYxS0feUPlO7zh7LkzWoA1VICF3PKRqcp3eM/Zc2e0AGuoAAu55SNTle/wnrPnzmgB1lABFnLLR6Yq3+E9Z8+d0QKsoQKs5ZaPvFD5Dn/j7PlptABrqADLueUjDyrf4S+dPT+NFmANFWBdt3xUvsP/6+z5abQAa6gA8MLZ84/+r7R/Z7QAa6gA8MLtK4/qM8BvVwHgmdtXXqnPAL9aBYAHt6/M1WeA36sCwIPbV+bqM8DvVQHgwe0rc/UZ4PeqAPC/bl95R30G+KUqADy4fWWuPgP8XhUAHty+MlefAX6vCgAPbl+Zq88Av1cFgGduX3mlPgP8ahUAXrh95VF9BvjtKgD8ye0r9RlgGRUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDN/AehvVOXxzRkPQAAAABJRU5ErkJggg==',
  'Squat_2': 'iVBORw0KGgoAAAANSUhEUgAAAwAAAAMACAIAAAAc45fZAAAgAElEQVR4AezBgWHjRpZo0ftC6BQmBTgEpyCFUC+WDeFVCK0UfgiLWCYEfo+99nSLZIFSiyAI3HMCSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSZIOJpAkSTqYQJIk6WACSZKkgwkkSUMvnN4IJO1IIEm65IUTZ94IJD2/QJJ05oUTV7wRSHpygSTpZy+cGPpG5wedRNJTCSRJP3vhxNA3Omc6yZ9OnIJA0oYFkqSfvXBi6BudM0XjTBBI2p5AkvSDF07c4BudHxSNK4JA0sYEkqSfvXBi6BudHxSNoSCQtCWBJOlnL5wY+kbnB0VjKAgkbUkgSfrZCyeGvtH5QdEYCgJJWxJIks68cOKKb3R+UDRuEAQ/+M7phbfgFUmPEEiSLnnhxJlvdM4UjaEg+NN3Tpx54S14RdKKAknS0AunN4I/NYozRWMoCOA7J6544S14RdJaAknSBzWKHxSNoSC+c2LohbfgFUmrCCRJv+zEiSs6HfhGY+iFt+AVSasIJElf4cSJM53On77RGHrhLXhF0ioCSdJXKIo/NVqn84NvNG7wwlvwiqT7CyRJX6EorvtGY+iFt+AVSasIJElfoSiu+0Zj6IW34BVJqwgkSV+hKK77RmPohbfgFUmrCCRJX6Eohr7RuOKFt+AVSWsJJElfpCjOJK3o/Ef/xv9y5oW34BVJKwokSV+kKP6WNM4Uv/Gnb/zvC2+d/0lmJK0ukCR9naKApHFF8Rs/SGYkrS6QJH2dopLGUPEbf0pmJD1CIEn6YieGit/4UzIj6RECSdIXOzFU/JbMSHqcQJL0y4r/k5y4SSDpcQJJ0qcUlyUnFgSSHiqQJH1EsSA5sSCQ9FCBJOkGxa2SEwsCSQ8VSJKGig9LTlwVSHq0QJJ0RfF5yYkLAkkbEEiSflb8kuSdEwSStiSQJP2t+FWJpCcQSNLhFV8gkfQ0Akk6sOJrJJKeSSBJh1R8jUTS8wkk6WCKL5NIekqBJB1G8WUSSU8skKQDKL5MIunpBZK0a8WXSSTtRCBJO1V8mUTSrgSStDvFl0kk7VAgSTtSfJlE0m4FkrQLxZdJJO1cIElPrvgyiaRDCCTpmRVfI5F0IIEkPafiaySSDieQpCdUfIFE0kEFkvRUil+V3Mf3E6+BpGcQSDqO0wvxxhN5OfEW/KD4JUnxGcnA9xPnXoPncvpOvCIdRiDpDl5OvAVbcXrhXLxxi6n4KnNyo5cTZ+ot+Kyk+HrJH76fuOY12L7Td87FK9LeBZK+zsuJc2/BI51euCbe+MtUPNac/OPlxBX1FnxQUtzP98bYa7Blp+9cE69IuxZI+iIvJ655Cx7j9MLYb7+zKf9qDNVbcJukuLfvjbHXYLNO3xmLV6T9CiR9hZcTY2/BA5xeGPvtdzblX42heguGkmI13xtjr8Fmnb4zFq9I+xVI+govJ8beggc4vTD22+9syr8aQ/UWXJIUK/veuMVrsE2n74zFK9J+BZK+wsuJsbdgbacXbvHb72zEvxo3qLfgT0nxWN8bY6/BNp2+c4t4RdqpQNIvezlxi7dgVdVo/2bst9/ZlH81xt462/G9MfYabFMV7Rtj8Yq0X4Gkr/ByYuwtWE81/tL+zdhvv7Mp/2qMvXW243tj7DXYoCr+0L4xFq9I+xVI+govJ8begjVU40ft34z99jub8q/G2FtA8UjJj76fuOY12Joq/tG+MRavSPsVSPp1Nb20maG34O6qca79m2vijb9MxWPNyT9eTlzzFlxVfKXkRq34vXHuNdiUKs61b1wTr0i7Fkj6dTUBL23mirfgvqox0P7NuXjjFlPxVebkRi8nzr0FG9SKf/ze+H+dP/RkU6q4pn3jXLwi7V0g6RfVxN9e2syZt+COqrEoO385vRBvbFsV/8j/d+L3INmuVpzryUZUMZbJX07fiVekwwgk/YqauOSlzW/B3VVjUXaeRxXvJJBsVCsu6skWVDGWiXRUgaRPq4lrcuauqnGL7DyVKn6U/CnZqFZc1JOHq2IsE+nAAkmfVhMX5cxdVeMW2XkqVbyT/CnZqFZc1JMHqmJRJtKxBZI+pyauyZk7qcaNsvNsqngn+VOyRa24piePUsWiTKTDCyR9Qk0M5Mw9VONG2Xk2VbyT/C3ZolZc05OHqOIWmUiHF0j6qJoYyJkvV43bZecJVfFO8rdki1pxTU9WVsWNMpEEgaSPqomBnPlC1fiQ7DynKt5J/pZsTisGerKmKm6UiaQ/BZI+pCYGcuYLVeNDsvOcqjiX/C3ZnFYM9GQ1VdwoE0l/CyR9SE0M5MyXqMZHZec5VXEu+UGyOa0Y6Mk6qrhRJpJ+EEi6XU2M5cyvq8aHZOeZVXEu+UGyOa0Y6MkKqrhRJpJ+Fki6UU2M5cyvq8aHZOeZVXFR8oNkW1ox0JMVVHGjTCSdCSTdqCbGcuZXVOOjsvPkqjiX/CzZllYM9OSuqrhdJpIuCSTdoibGcuZXVONDsvP8qrgo+VmyLa0Y6Mn9VHG7TCRdEUhaVBOLcuZzqvEJ2Xl+VZxLziQb0oqxntxJFbfLRNJ1gaRFNbEoZz6hGp+QnV2o4lxyJtmQVoz15B6quF0mkoYCSWM1sShnPqEaH5WdvajiouRMsiGtGOvJl6vidplIWhJIGquJRTnzUdX4qOzsSBXnkkuSDWnFWE++VhW3y0TSDQJJAzWxKGc+pBqfkJ0dqeKi5JJkQ1ox1pOvUsWHZCLpNoGkgZpYlDO3q8ZHZWdfqrgouSLZilaM9eSrVPEhmUi6WSDpmppYlDM3qsYnZGd3qrgouSLZilaM9eRLVPEhmUj6iEDSRTVxi5y5RTU+ITu7U8U1yRXJVrRirCe/rooPyUTSBwWSLqqJRTlzi2p8Qnb2qIprkiuSrWjFWE9+URUfkomkjwsknauJW+TMomp8VHb2q4qLMqG4LNmEVizqya+o4kMykfQpgaRzNbEoZ8aq8QnZ2a8qrkmuSLaiFWM9+bQqPioTSZ8VSHqnJm6RMwPV+ITs7FcV12RCcVmyFa0Y68mnVfEhmUj6BYGkd2piUc5cU43Pyc6uVXFNJhSXJVvRirGefEIVH5WJpF8TSPpRTdwiZy6qxidk5wCquCYTisuSrWjFWE8+qoqPykTSLwsk/agmFuXMRdX4hOwcQBXXZPIfxWXJVrRirCcfUsVHZSLpKwSS/lETt8iZc9X4qOwcRhXXZPIfxWXJJrRiUU9uV8WHZCLp6wSS/lETi3LmnWp8QnYOo4qBTP6juCzZhFaM9eR2VXxUJpK+TiDpLzVxi5z5UTU+ITtHUsU1mfxHcVWyCa0Y68mNqviQTCR9tUDSX2piUc78qBqfkJ0jqWIgk/8orko2oRVjPblFFR+SiaQ7CCT9oSZukTP/qMZHZed4qhjI5D+Kq5JNaMVYTxZV8SGZSLqPQNIfamJRzvylGp+QneOpYiCT/1NclTxeK8Z6sqiKD8lE0t0EkmpiUc78pRoflZ2jqmIgk/9TXJU8XivGejJWxYdkIumeAkk1sShn/lCNj8rOUVUxlsn/Ka5KHq8VYz0ZqOJDMpF0Z4F0cDWxKGeq8QnZObAqBjL5r+Kq5PFaMdaTa6q4XSaSVhFIB1cTyyY+ITvHVsU1mfxXMZI8XivGenJRFbfLRNJaAunIamJRzlTjQ7JzeFUMZPJfxVXJJrRioCcXVXG7TCStKJCOrCaWTXxIdg6virFM/qu4KtmEVgz05FwVt8tE0roC6chqYsHE7bKjP1UxkMlPiquSTWjFQE/eqeJ2mUhaXSAdVk0smLhddvS3KgYy+UlxVfJ4rRjoyTtV3C4TSY8QSIdVEwsmbpQd/a2KsUx+UlyVPF4rBnryoypulImkxwmkY6qJBRO3yI5+VsVAJj8pRpLHa8VAT/5RxY0ykfRQgXRMNTEycYvs6GdVjGXyk2IkebxWDPTkL1XcKBNJjxZIx1QTIxNj2dGZKsYyea8YSR6vFdf05C9V3CgTSRsQSAdUEyMTA9nRFVWMZfJeMZI8Xiuu6ckfqrhRJpK2IZAOqCZGJq7Jjq6rYiyT94qR5PFacU1PqrhRJpI2I5COpiZGJi7KjoaqGMvkgmIkebxWXNSTKm6RiaSNCaSjqYmrJi7KjpZUMZbJBcVI8nituKgnVSzKRNL2BNLR1MRlE+eyoxtUsSiTC4qR5PFacdHEskwkbVIgHUpNXDXxTnZ0myrGMrmgWJA8XivOTSzIRNKGBdKhnKBPXDDxo+zoI6oYy+SCYkHyeK14Z2JBJpK2LZCO4MQFfeL/TPwjO/qgKsYyuawYSTahFX+oRnb+MLEgE0mbF0i7d+KqPsHEP7Kjj6tiLJPLipHk8U4nzvXORZlIehKBtG8nFvTGH7KjT6liUSaXFSPJg51OXNM772Qi6XkE0r6dWBDoV1QxlslVxUjySKcTY73zl0wkPZtA2rcTCwJ9WhWLMrmqGEke6XRirHcykfScAmnHTtwk0OdUMZbJSDGSPNLpxFgEkp5WIO3biQWBPq2KsUxGipHkYU4nbhGBpOcUSPt2YkGgz6liUSZXFQuSRzqdGItA0tMKpH07sSDQ51QxlslIsSB5pNOJsQgkPa1A2r0TVwX6nCoWZTJSLEge6XRiLAJJTyuQjuDEBYE+p4pFmSwoRpLHO524JgJJzyyQDuUEgX5RFYsyWVCMJJtwOnEuAklPLpCkD6piUSYLipFkW04nIpC0F4EkfVAVY5ksK0YSSbqfQJI+oopFmSwrRhJJup9Akm5WxaJMlhULEkm6n0CSblbFWCY3KRYkknQ/gSTdpopFmdykWJBI0v0EknSbKhZlcpNiQSJJ9xNI0m2qWJTJTYoFiSTdTyBJt6liLJNbFQsSSbqfQJJuUMWiTG5VLEgk6X4CSVpSxaJMPqBYkEjS/QSStKSKRZncqliQSNJdBZI0VMUtMrlVsSCRpLsKJGmoikWZfECxIJGkuwokaaiKRZl8QLEgkaS7CiRpqIqxTD6mWJBI0l0FknRdFYsy+ZhiQSJJdxVI0hVVLMrkw4oFiSTdVSBJV1SxKJMPKxYkknRXgSRdUsUtMvmwYkEiSXcVSNIlVSzK5MOKBYkk3VsgSWequEUmH1YsSCTp3gJJOlPFokw+o1iQSNK9BZJ0popFmXxGsSCRpHsLJOlMFWOZfFKxIJGkewsk6WdVLMrkk4oFiSTdWyBJP6jiFpl8UrEgkaR7CyTpB1XcIpNPKhYkknRvgST9rYpbZPJ5xYJEku4tkKS/VbEok88rliWSdG+BJP2tikWZfF6xLJGkewsk6W9VjGXyS4oFiSStIJCkP1WxKJNfUixIJGkFgST9qYpFmfySYkEiSSsIJAmquEUmv6RYkEjSCgJJh1fFLTL5VcWCRJJWEEg6vCpukcmvKhYkkrSCQNLhVbEoky9QLEgkaQWBpMOrYiyTL1AsSyRpBYGkY6tiUSZfoFiWSNIKAknHVsWiTL5AsSCRpHUEkg6siltk8gWKBYkkrSOQdGBVLMrkaxQLEklaRyDpqKq4RSZfo1iQSNI6AklHVcWiTL5MsSCRpHUEko6qikWZfJliQSJJ6wgkHVIVizL5SsWCRJLWEUg6pCrGMvlixYJEktYRSDqeKhZl8sWKBYkkrSOQdDxVLMrkKxULEklaTSDpYKq4RSZfqViQSNJqAkkHU8WiTL5YsSCRpNUEkg6mikWZfLFiQSJJqwkkHUkVizL5esWCRJJWE0g6kirGMrmLYkEiSasJJB1GFYsyuYtiQSJJqwkkHUYVizK5i2JBIkmrCSQdRhWLMrmLYkEiSasJJB1GFWOZ3EWxIJGkNQWSjqGKRZncRbEgkaQ1BZIOoIpFmdxLsSCRpDUFkg6girFM7qhYkEjSmgJJe1fFokzuqFiQSNKaAkl7V8WiTO6oWJBI0poCSXtXxaJM7qhYkEjSmgJJu1bFokzuq1iQSNKaAkm7VsVYJndXjCSStLJA0n5VsSiTuytGEklaWSBpv6oYy2QNxUgiSSsLJO1XFWOZrKEYSSRpZYGknapiUSZrKEYSSVpZIGmnqhjLZA3FgkSSVhZI2qMqFmWyhmJBIkkrCyTtURVjmaykWJBI0soCSXtUxVgmKylGEklaXyBpd6oYy2Q9xUgiSesLJO1OFWOZrKcYSSRpfYGkfaliUSbrKUYSSVpfIGlfqhjLZFXFSCJJ6wsk7UsVY5msqhhJJGl9gaQdqWIsk7UVI4kkrS+QtCNVjGWyqmJBIknrCyTtSBVjmayqWJBI0voCSXtRxVgmaytGEkl6iEDSXlQxkMkDFCOJJD1EIGkXqhjL5AGKkUSSHiKQtAtVDGTyGMVIIkkPEUjahSoGMnmMYiSRpIcIJD2/KsYyeYxiJJGkhwgkPb8qBjJ5mGIkkaSHCCQ9vyquyeSRipFEkh4ikPTkqhjI5JGKqxJJepRA0jOrYiyTRyquSiTpUQJJz6yKgUwerLgqkaRHCSQ9syoGMnmw4qpEkh4lkPS0qhjI5PGKqxJJepRA0tOqYiCTBytGEkl6lEDS06rimkwer7gqkaQHCiQ9pyoGMnm84qpEkh4okPScqrgmk00orkok6YECSc+pimsy2YTiqkSSHiiQ9ISquCaTrSiuSiTpgQJJT6iKazLZiuKqRJIeKJD0hKq4JpOtKK5KJOmBAknPpoprMtmQ4rJEkh4rkPRsqrgok20pLksk6bECSU+limsy2ZbiskSSHiuQ9FSquCaTbSkuSyTpsQJJT6WKizLZnOKyRJIeK5D0VKq4KJPNKS5LJOmxAknPo4qLMtmi4rJEkh4rkPQ8qjiXyRYVlyWS9HCBpCdRxUWZbFFxWSJJDxdIehJVnMtko4rLEkl6uEDSk6jiXCYbVVyWSNLDBZKeQRUXZbJRxWWJJD1cIOkZVHEuk+0qLksk6eECSZtXxUWZbFdxQSJJWxBI2rwqzmWyacUFiSRtQSBp86o4l8mmFRckkrQFgaRtq+JcJltXXJBI0hYEkratincyeQLFBYkkbUEgaduqeCeTJ1BckEjSFgSStq2KdzJ5AsUFiSRtQSBpw6p4J5PnULyXSNJGBJI2rIp3MnkOxXuJJG1EIGnDqvhRJk+jeC+RpI0IJG1VFe9k8jSK9xJJ2ohA0lZV8aNMnknxk0SStiOQtFWnF/rv/CWTJ1P8JJGk7QgkbczphXPxxpMpfpJI0nYEkrbk9MI18cYzKX6SSNJ2BJI24/TCWLzxNIqfJJK0HYGkzTi9MBZvPI3ivxJJ2pRA0macXhiLN55G8V+JJG1KIGkbTi/cIt54DsUfvjdeOySStCmBpM04vTAWbzyF7yfOvQaStBGBpM04vTAWb2zf9xPXvAaStAWBpM04vTAWb2zc9xNjr4EkPVwgaUtOL1wTb2zf9xNjr4EkPVwgaWNOL5yLN57C9xNjr4EkPVwgaatOL/Rv/CM7G/f9xC1eA0l6rEDSVlXjR9nZvu8nxl4DSXq4QNJWVeNH2dm+7yfGXgNJerhA0lZV453sbNz3E2P/05kTSXqsQNJWVeOd7Gzf9xPX/E/nL3MiSQ8USNqqaryTnafw/cS5/+n8aE4k6VECSVtVjXey81y+n3gN/jAV78yJJD1KIGmrqnEuO09qKt6ZE0l6iEDSVlXjXHae1FScmxNJWl8gaauqcS47z2sqzs2JJK0skLRV1TiXnec1FRfNiSStKZC0VdW4KDvPayoumhNJWk0gaauqcVF2ntdUXDQnkrSaQNJWVeOi7Dy1qbhoTiRpHYGkDavGuew8tam4Zk4kaQWBpA2rxkXZeWpTcc2cSNK9BZI2rBoXZefZTcVFcyJJ9xZI2rBqXJSdZzcV18yJJN1VIGnDqnFRdnZgKq6ZE0m6n0DShlXjmuw8u6kYmBNJupNA0oZV45rs7MBUXDMnknQngaQNq8Y12dmBqRiYE0m6h0DStlXjouzsw1QMzIkkfblA0rZV45rs7MNUDMyJJH2tQNK2VeOa7OzDVAzMiSR9rUDStlXjmuzsxlQMzIkkfaFA0rZV45rs7MlUDMyJJH2VQNK2VWMgO7sxFQNzIklfJZC0bdUYyM6eTMXAnEjSlwgkbVs1BrKzJ1MxNieS9OsCSdtWjYHs7MxUjM2JJP2iQNLmVeOa7OzPVAzMiST9okDS5lXjmuzsz1SMzYkk/YpA0uZVYyA7OzMVi+ZEkj4tkLR51RjIzv5MxaI5kaTPCSRtXjUGsrNLU7FoTiTpEwJJm1eNgezs0lQsmhNJ+oRA0uZVYyA7ezUVi+ZEkj4qkPQMqjGQnb2airE5kaSPCiQ9g2oMZGfHpmJsTiTpQwJJz6AaA9nZsalYNCeSdLtA0jOoxkB29m0qFs2JJN0okPQMqjGWnR2bikVzIkk3CiQ9g2qMZWffpmLRnEjSLQJJT6IaA9nZt6m4xZxI0qJA0pOoxkB2dm8qbjEnkjQWSHoS1RjIzhFMxaI5kaSxQNKTqMZYdo5gKhbNiSQNBJKeRDXGsnMEU3GLOZGkawJJT6IaY9k5gqm4xZxI0jWBpCdRjbHsHMRU3GJOJOmiQNLzqMZYdg5iKm4xJ5J0LpD0PKoxlp3jmIpFcyJJ5wJJz6MaY9k5lKlYNCeS9E4g6XlUYyw7hzIVt5gTSfpRIOl5VGMsO4cyFTeaE0n6RyDpeVRjLDtHMxU3mhNJ+ksg6alUYyA7BzQVt5gTSfpLIOmpVGMsOwc0FYvmRJL+Ekh6KtUYy84BTcUt5kSS/hBIeirVGMvOMU3FLeZEkgJJT6UaY9k5rKm4xZxIOrhA0lOpxlh2DmsqbjEnkg4ukPRUqrEoO4c1FbeYE0lHFkh6NtUYy85hTcWN5kTSYQWSnk01xrJzZFNxozmRdEyBpGdTjbHsHNxU3GJOJB1TIOnZVGMsO5qKRXMi6ZgCSc+mGouyo6lYNCeSDiiQ9ISqMZYdTcUt5kTS0QSSnlA1xrKjqbjRnEg6lEDSE6rGWHb0h6m40ZxIOo5A0hOqxlh29JepuMWcSDqOQNITqsZYdvSXqbjRnEg6iEDSE6rGouzoL1NxozmRdASBpOdUjbHs6B9TcYs5kXQEgaTnVI2x7OhHU7FoTiQdQSDpOVVjLDt6ZyoWzYmk3QskPadqjGVH70zFLeZE0r4Fkp5TNRZlRz+aihvNiaQdCyQ9rWqMZUfvTMWN5kTSXgWSnlY1xrKjc1NxizmRtFeBpKdVjbHs6KKpuMWcSNqlQNLTqsZYdnTRVNxiTiTtUiDpaVVjLDu6ZipuMSeS9ieQ9MyqMZAdDUzFLeZE0s4Ekp5ZNcayo4GpWDQnknYmkPTMqjGWHQ1MxS3mRNKeBJKeWTXGsqOBqbjRnEjajUDSM6vGWHY0NhU3mhNJ+xBIembVGMuOFk3FLeZE0j4Ekp5ZNcayo1tMxaI5kbQPgaQnV42x7OgWU7FoTiTtQCDpyVVjLDu6xVTcYk4kPbtA0pOrxlh2dKOpuMWcSHpqgaQnV42x7Oh2U3GLOZH0vAJJz68aA9nR7abiFnMi6XkFkp5fNQayow+ZikVzIul5BZKeXzUGsqOPmopFcyLpSQWSnl81xrKjD5mKW8yJpGcUSHp+1RjLjj5qKm4xJ5KeTiBpF6oxkB19wlQsmhNJTyeQtAvVGMiOPmcqFs2JpOcSSNqFagxkR58zFYvmRNJzCSTtQjUGsqNPm4pFcyLpiQSS9qIaA9nRp03F2JxIeiKBpL2oxkB29CumYmxOJD2LQNJeVGMgO/oVU7FoTiQ9hUDSXlRjIDv6RVOxaE4kbV8gaS+qMZAd/aKpWDQnkrYvkLQj1bgmO/p1U7FoTiRtXCBpR6pxTXb0JaZi0ZxI2rJA0o5UYyA7+hJTMTYnkrYskLQj1RjIjr7KVAzMiaQtCyTtSDUGsqOvMhVjcyJpswJJO1KNgezoC03F2JxI2qZA0r5U45rs6GtNxcCcSNqmQNK+VOOa7OjLTcXAnEjaoEDSvlTjmuzoy03FwJxI2qBA0r5U45rs6B6mYmBOJG1NIGl3qnFRdnQnUzEwJ5I2JZC0O9W4Jju6h6kYmBNJmxJI2p1qXJMd3clUDMyJpO0IJO1ONa7Jju5nKgbmRNJGBJL2qBoXZUd3NRXXzImkjQgk7VE1LsqO7moqBuZE0hYEkvaoGhdlR/c2FdfMiaQtCCTtUTUuyo5WMBXXzImkhwsk7VQ1zmVH65iKi+ZE0sMFknaqGhdlRyuYimvmRNJjBZJ2qhoXZUfrmIpr5kTSAwWSdqoaF2VHq5mKi+ZE0gMFkvarGueyozVNxUVzIulRAkn7VY1z2dGapuKiOZH0KIGk/arGuexoZVNx0ZxIeohA0n5V41x2tL6pODcnkh4ikLRr1XgnO3qIqTg3J5LWF0jatWq8kx09ylS8MyeS1hdI2rVqvJMdPYsVYRQAAAxvSURBVNBUvDMnklYWSNq1aryTHT3QVLwzJ384nYhA0joCSXtXjR9lR481Ff/438a5CCTdVSBp76rxo+zo4abiD//buCYCSfcTSNq7avyjdQJtwunEWASS7iSQdAAnLgj0SKcTYxFIupNA0t6duCrQw5xOjEUg6U4CSbt2YkGgBziduEUEku4hkLRrJxYEeozTibEIJN1JIGnXTiwI9BinE2MRSLqTQNJ+nbhJoAc4nRiLQNKdBJJ27cSCQA9zOnFNBJLuJ5C0aycWBHqk04lzEUi6q0DSrp1YEOjxWlGN7PylJ5LuKpC0dyeuCrQJrfhRTyTdVSDpAE5cEGgTWvFOTyTdVSDpGGriD22mT/whZ7QRrXinJ5LuKpB0ADXxTs5oI1rxTk8k3VUg6QBq4p2c0Ua04p2eSLqrQNIB1MQ7OaMtaMW5nki6q0DSAdTEOzmjLWjFuZ5IuqtA0gHUxDs5oy1oxbmeSLqrQNIB1MQ7OaMtaMW5nki6q0DS3tXEuZzRFrTiXE8k3VUgae9q4lzOaAtaca4nku4qkLR3NXEuZ/RwrbioJ5LuKpC0dzVxLmf0cK24qCeS7iqQtHc1cS5n9HCtuKgnku4qkLR3NXEuZ/RwrbioJ5LuKpC0dzVxLmf0cK24qCeS7iqQtGs1cVHO6LFacVFPJN1bIGnXauKinNFjteKinki6t0DSrtXERTmjx2rFRT2RdG+BpF2riYtyRo/Viot6IuneAkm7VhMX5YweqxUX9UTSvQWSdq0mLsoZPVYrLuqJpHsLJO1aTVyUM3qgVlzTE0n3FkjatZq4KGf0QK24pieS7i2QtF81cU3O6IFacU1PJN1bIGm/auKanNEDteKanki6t0DSftXENTmjB2rFNT2RdG+BpP2qiWtyRg/Uimt6IuneAkn7VRPX5IweqBXX9ETSvQWS9qsmrskZPUorBnoi6d4CSftVE9fkjB6lFQM9kXRvgaT9qolrckaP0oqBnki6t0DSftXENTmjR2nFQE8k3Vsgab9q4pqc0aO0YqAnku4tkLRfNXFNzuhRWnFNTyStIJC0UzUxkDN6lFZc0xNJKwgk7VRNDOSMHqIVAz2RtIJA0k7VxEDO6CFaMdATSSsIJO1UTQzkjB6iFQM9kbSCQNJO1cRAzughWjHQE0krCCTtVE0M5IweohUDPZG0gkDSTtXEQM7oIVox0BNJKwgk7VRNDOSMHqIVAz2RtIJA0k7VxEDO6CFaMdATSSsIJO1UTQzkjNbXirGeSFpBIGmnamIgZ7S+Voz1RNIKAkk7VRMDOaP1tWKsJ5JWEEjao5oYyxmtrxVjPZG0gkDSHtXEWM5ofa0Y6ImkdQSS9qgmxnJG62vFQE8krSOQtEc1MZYzWl8rBnoiaR2BpD2qibGc0fpaMdATSesIJO1RTYzljFbWirGeSFpHIGmPamIsZ7SyVoz1RNI6Akl7VBNjOaOVtWKsJ5LWEUjao5oYyxmtrBVjPZG0jkDSHtXEWM5oZa0Y64mkdQSS9qgmxnJGK2vFWE8krSOQtEc1MZYzWlkrBnoiaTWBpD2qibGc0cpaMdATSasJJO1RTQzkjFbWirGeSFpNIGmPamIgZ7SyVoz1RNJqAkm7UxNjOaOVtWKsJ5JWE0janZoYyxmtrBVjPZG0mkDS7tTEWM5oZa0Y64mk1QSSdqcmxnJGK2vFWE8krSaQtDs1MZYzWlkrxnoiaTWBpN2pibGc0cpaMdYTSasJJO1OTYzljNbUikU9kbSaQNLu1MRYzmhNrRjriaQ1BZJ2pybGckZrasVYTyStKZC0OzUxljNaUyvGeiJpTYGk3amJsZzRmlox1hNJawok7U5NjOWM1tSKsZ5IWlMgaXdqYixntKZWjPVE0poCSbtTE2M5ozW1YqwnktYUSNqdmhjLGa2pFWM9kbSmQNLu1MRYzmhNrRjriaQ1BZJ2pybGckaracWinkhaUyBpd2piIGe0plaM9UTSygJJu1MTAzmjNbVirCeSVhZI2peaGMsZrakVYz2RtLJA0r7UxFjOaE2tGOuJpJUFkvalJsZyRmtqxVhPJK0skLQvNTGWM1pTK8Z6ImllgaR9qYmxnNGaWjHWE0krCyTtS02M5YzW1IqxnkhaWSBpX2piLGe0plYM9ETS+gJJ+1ITYzmjNbVioCeS1hdI2peaGMsZraYVYz2RtL5A0r7UxFjOaDWtGOuJpPUFkvalJsZyRqtpxVhPJK0vkLQvNTGWM1pNK8Z6Iml9gaR9qYmxnNFqWjHWE0nrCyTtS00M5IzW1IqxnkhaXyBpX2piIGe0plYM9ETSQwSS9qUmBnJGa2rFQE8kPUQgaV9qYiBntKZWDPRE0kMEkvalJgZyRmtqxUBPJD1EIGlfamIgZ7SaVoz1RNJDBJL2pSYGckaracVYTyQ9RCBpX2piIGe0mlaM9UTSQ/z/9uCgyGHlDAPg90MIFmHxQtBgGkGwsQhLIDiHd0mtbVUOG8k1090VYCx9yYG2h9OsPce2FuASFWAsfcmBtofTrD0HthbgKhVgLH3JgbaH06w9B7YW4CoVYCx9yYG2h9OsPQe2FuAqFWAsfcknbQ9nWnsObC3AVSrAWPqST9oezrT2HNhagKtUgLH0JZ+0PZxp7TmwtQBXqQBj6Us+aXs409pzYGsBrlIBxtKXfNL2cKa158DWAlylAoylL/mk7eFMa88nWwtwoQowlr7kk7aHM609n2wtwIUqwFj6kk/aHk6z9hzYWoALVYCx9CWftD2cZu05sLUAF6oAY+lLPml7OM3ac2BrAS5UAcbSl3zS9nCatefA1gJcqAKMpS95q+3hTGvPga0FuFAFGEtf8lbbw5nWnk+2FuBaFWAsfclbbQ9nWns+2VqAa1WAsfQlb7U9nGnt+WRrAa5VAcbSl7zV9nCmteeTrQW4VgUYS1/yVtvDmdaeT7YW4FoVYCx9yVttD2daez7ZWoBrVYCx9CVvtT2cae35ZGsBrlUBxtKXvGp7ONna89bWAlyuAoylL3nV9nCyteetrQW4XAUYS1/yqu3hZGvPW1sLcLkKMJa+5FXbw8nWnre2FuByFWAsfcmrtoeTrT1vbS3A5SrAWPqSV20PJ1t73tpagMtVgLH0Ja/aHs609ry1tQDfoAKMpS/5pe3hZGvPW1sL8A0qwFj6kl/aHk629ry1tQDfoAKMpS/5pe3hZGvPW1sL8A0qwFj6kl/aHk629ry1tQDfoAKMpS/5pe3hZGvPq60F+BIVYCx9yS9tDydbe15tLcCXqABj6Uv+se63bXm0PZxv7Xm1tQBfogKM5ZlbXlQe4URrz6utBfgSFWAgz9zyQeURzrL2vNpagC9RAUbxzC2HKo9wirXn1dYCfIkKMIpnbjlUeYRTrD2/bC3A96gAo3jmlkOVRzjF2vPL1gJ8jwowhGdu+R9UHuH/b+35ZWsBvkcFGMUztxyqPMIp1p5fthbge1SAUTxzy6HKI5xi7flvWwvwVSrAKJ655VDlEU6x9vyj/yvt39lagK9SAQbyzC0fVB7hLM97XtVPgC9RAcbyzC0vKo9wluc9n9RPgG9QAQb1zK3yCOd63nOsfgJcrgLA33nec6x+AlyuAsDfed5zrH4CXK4CwB953vO/qJ8A16oA8Hee9xyrnwCXqwDwd573HKufAJerAPB3nvccq58Al6sA8Kee93xSPwG+QQWAv/a851X9BPgSFQD+b5731E+Ab1MBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYTAUAYDIVAIDJVAAAJlMBAJhMBQBgMhUAgMlUAAAmUwEAmEwFAGAyFQCAyVQAACZTAQCYzH8Ah3MZl13RfLUAAAAASUVORK5CYII=',
  'Squat_3': 'iVBORw0KGgoAAAANSUhEUgAAAwAAAAMACAIAAAAc45fZAAAgAElEQVR4AezBC2Hc6Lp22/lBCIVNQQ1hUbAh6IVwMGwIjyAkFH4IS1g2hDqd7nQnsaskuW7WZY7RkCRJOpiGJEnSwTQkSZIOpiFJknQwDUmSpINpSJIkHUxDkiTpYBqSJEkH05AkSTqYhiRJ0sE0JEmSDqYhSZJ0MA1JkqSDaUiSJB1MQ5Ik6WAakiRJB9OQJEk6mIYkSdLBNCRJkg6mIUmSdDANSZKkg2lIkiQdTEOSJOlgGpIkSQfTkCRJOpiGJEnSwTQkSZIOpiFJknQwDUmSpINpSJIkHUxDkiTpYBqSJEkH05AkSTqYhiRJ0sE0JEmSDqYhSZJ0MA1JkqSDaUiSJB1MQ5Ik6WAakiRJB9OQJEk6mIYkSdLBNCRJkg6mIUmSdDANSZKkg2lIkiQdTEOSJOlgGpIkSQfTkCRJOpiGJEnSwTQkSZIOpiFJknQwDUmSpINpSJIkHUxDkiTpYBqSJEkH05AkSTqYhiRJ0sE0JEmSDqYhSZJ0MA1JkqSDaUiSJB1MQ5Ik6WAakiRJB9OQJEk6mIYkSdLBNCRJkg6mIUmSdDANSZKkg2lIkiQdTEOSJOlgGpIkSQfTkCRJOpiGJEnSwTQkSZIOpiFJknQwDUmSpINpSJIkHUxDkiTpYBqSJEkH05AkSTqYhiRJ0sE0JEmSDqYhSZJ0MA1JkqSDaUiSJB1MQ5Ik6WAakiRJB9OQJEk6mIYkSdLBNCRJkg6mIUmSdDANSZKkg2lIktbqhdM3GpLurSFJWpkXTrzzjYakO2lIktbkhRMXfKMh6R4akqTVeOHEpC8MLDNQSLqgIUlajRdOTPrCwMcNFJJ+0ZAkrcYLJyZ9YeAGA8UvTpwaDel4GpKkdXjhxAJfGLhN6Hmn0ZAOoyFJWo0XTkz6wsBtQs8FjYZ0DA1J0mq8cGLSFwZuEHomNRrSATQkSavxwolJXxi4QeiZ1GhIB9CQJK3JCycu+MLAbULPpEZDOoCGJGllXjjxzjca0BOuFXoWaDSkvWtIktbqhdM3Guf0hI8LPZMaDekAGpKkjesJy4SeSY2GdAANSdJhnDgxqdGQDqAhSTqSEycuGBiAopD2riFJOpgTJ94ZGPhHUUi71pAkHU8I0NMPDLxTFNKuNSRJxxPCpKKQ9qshSTqeECYVxQInvjZekbamIUk6nhAmFcVlJ77yTuMVaSMakqTjCWFOUZxz4isXNF6RtqAhSTqeEOYUxTsnvjKp8Yq0eg1J0iGFMKko3jnxlUmNV6TVa0iSDimEOUXxj9ABPf8fkxqvSKvXkCQdUghzigJCx196/j8WaLwirVtDknRUIcwY+E3X8x/+8Y2XF77xu8Yr0uo1JElHFcKMgd90Pf/5xgvvvPCNvzRekVavIUk6qhBmDPym+0K44IVvQOMVafUakqQDC2HKwC++8F8mvdKQtqAhSTqwEN4p+jDw3cAvvvBfJr3SkLagIUk6thD+UvS8E/7gH1/4L5NeaUhb0JAkHV5I0XNB+AP4wn9Z4JWGtHoNSZI4MSn8AXzhv0x6pSFtQUOSJE5MCn8AX/gvk15pSFvQkCSJE5PCH8AX/sukVxrSFjQkSUd3YoHwRzF+5cQFrzSkjWhIkg4p/FScmNH4x1dOvPNKQ9qOhiTpYMJbxYkZjXe+cnqlIW1QQ5J0GOG84sSMhrQjDUnSAYQZxYmLGtK+NCRJuxYWKf504oyGtDsNSdJOhaWKN07QkParIUnanfABhXQ4DUnSjoSPKaQjakiS9iJ8QCEdV0OStH3hYwrp0BqSpC0LH1NIoiFJ2qbwYYWk7xqSpA0KH1ZI+qEhSdqU8GGFpN80JEkbEa5RSHqrIUlavXClQtIZDUnSuoVrFJIuakiS1ipcqZA0pSFJWp9wpULSvIYkaU3C9QpJizQkSasRrlRI+oCGJGkdwpUKSR/TkCR9tnC9QtKHNSRJnydcr5B0pYYk6TOE6xWSbtKQJD1duF4h6VYNSdIThesVku6jIUl6inCTQtLdNCRJjxeuV0i6s4Yk6cHC9QpJ99eQJD1MuF4h6VEakqQHCDcpJD1QQ5J0b+F6haSHa0iS7idcr5D0JA1J0p2E6xWSnqchSbpZuEkh6akakqTbhOsVkj5BQ5J0rXC9QtKnaUiSrhKuV0j6TA3puV5OfGtImxauV0j6fA3pKV5OvPetIW1LuF4haS0a0uO9nLjkW0PainClQtK6NKQHezkx7VtDWr9wpULS6jSkB3s5Me1bQ1qzcL1C0ho1pAd7OTHtW0NarXClQtJ6NaRHejmxxLeGtDbhSoWktWtID/ZyYtq3hrQ24RqFpG1oSA+V7qUfmfStoSudXmjf0F2FKxWSNqMhPVS6l35k0reGPub0wnvtG7pZuEYhaWMa0uOk4y8v/cgF3xr6mNMLl7Rv6AbhwwpJm9SQHiQdv3jpR9751tDHnF6Y1r6hjwsfVkjasIb0COk4q8aXE98autLphWl//IfbjcWRhA8rJG1bQ3qEdLxXI7rR6YVpf/yHZxqLjQsfU0jag4Z0d+k4q0Z0i9MLS/zxH1ZiLFYsfEwhaT8a0n2l45Ia0Y1OL0z74z+s31h8tvABhaS9aUj3lY6zakRXS8/f+v9j2h//YYvG4lnCxxSSdqgh3VE6LqkRXSE9v+r/j2l//Id9GIsHCB9QSNqthnQv6bikRvQh6bmk/z8uad/4Wxf2ZyxuE5YqJO1cQ7qXdFxSI1ooPbP6/+O99o33urBXYzHt5cS3xj/CIoWkQ2hId5GOS2pEs9KzRA386vRC+8ZyXdifsfjVy4l38q0xp5B0IA3pdumYUCOalp5ZNfAIXdiT/+m5IN8aFxSSDqch3S4dl9SILknPQjXwNF3YqP/pmZRvjd8V4a1C0gE0pBulY0KN6L30LFcDn6sLm/A/PZPyrfGPItxBIWmbGtIt0jGhRvRGepargTXrwqr8T8+kfGtFeKpC0io1pFukY0KN6G/p+ZAa2LouPNP/9CzxbWAVCkmfqiFdLR3TakTp+agaOJQu3MX/9Ez7NrBqhaRnaUjXSce0GlF6PqoGdJ2XE9O+DWxMIekxGtJ10jGtRo4sPR9VA7rFy4lp3wY2r5B0Dw3pCumYViPHlJ4r1IDu4uXEJd8avwk7UUj6uIZ0hXRMq5GjSc8VakD39XLivW+NKWEniiW+nnhtSMfWkD4qHdNq5GjSc4Ua0L0l/K3+34n/NIoPCjtRvPH1xHuvDemQGtJHpWNajRxEeq5TA3qAhF8VUNwsbF7x9cQlrw3peBrSh6RjWo0cRHquUAN6jIRfFVA8RtiWrz3TXhvSwTSk5dIxq0b2LT1XqwE9TMK/ir8UzxLW7GvPtNeGdDANabl0TKuRHUvP1WpAD5PwRvGX4lOFlfjaM+21IR1MQ1ooHbNqZJfSc7Ua0IMlvFH8pVil8Exfe5Z4bWiJ01faK9q+hrREOmbVyC6l5zo1oMdLeKP4R7FN4b6+9kx7bWja6SvvtVe0WQ1piXTMqpE9Sc8takCPl/BG8Y/iGMKU4k9fT0x7bWjC6SuXtFe0TQ1pVjpm1ciepOdqNaBnSfhV8YtCP3w9Me21oUtOX5nWXtEGNaRZ6ZhVIzuQnlvUgJ4o4Y3iF4V++nrikteGJpy+Mq29og1qSNPSMatGdiA9t6gBPVHCe8UvCv3Uh//0vPfa0LTTV6a1V7RBDWlCOpaoke1Kz41qQM+V8F7xi0I/9eFX/+n5fwNDoVmnryzRXtHWNKQJ6ZhVIxuVnhvVgD5DwhvF7wr91If3hkKXJPyr/8K09oo2qCFdko4lamSL0nOLGtAnSXiv+F2hn/rw3lDorIRf9V+Y1l7RBjWkS9Ixq0a2JT23qwF9koT3incK/dCHs4ZCv0o4q//CtPaKNqghXZKOaTWyLem5UQ3o8yS8V5xT6Ic+vDcU+lfCtP4Ll7RXtE0N6ax0zKqRrUjPjWpAny3hveKcQt/14ayh0N8SplXxp9NX3muvaLMa0nvpWKJGVi49t6sBrUPCe8U7hX7ow1lDoYRZVbxx+kp7RdvXkN5Lx6waWbn03K4GtA4J7xXnFPqhD2cNxZElLFGF9qshvZGOJWpktdJzoxrQmiScVZxT6Ls+XDIUh5WwRBXatYb0Rjpm1cg6ped2NaA1STirCsIZhb7rwyVDcUAJS1ShA2hIv0rHrBpZofTcrga0PgnvVUE4o9APfbhkKA4lYaEqdAwN6V/pWKJG1iY9N6oBrVLCe1V8F84o9F0fLhmK40hYrgodRkP6Vzpm1ciqpOdGNaC1Sjiriu/CGYW+68MlQ3EQCQtVoYNpSH9LxxI1sgbpuYsa0FolnFXFd+GMQj/04ZKh2L2E5arQ8TSkv6VjVo2sQXpuVwNasYRLqvgunFHouz5MGIodS/iQKnRIDelP6ViiRj5Xem5UA1q9hEuq+CGcUei7PkwYil1K+KgqdFQN6U/pmFUjnyU9d1ED2oKES6r4LpxX6Ls+TBiK/Un4kCp0bA0pHbNq5LOk5y5qQFuQMKGK78J5hejDtKHYk4SPqkKH19DBpWOJGnm+9NxFDWg7Ei6p4odwRqHv+jBhKHYj4QpVSNDQwaVjVo08X3puVAPamoRLqvghnFfouz5MGIp9SPioKqR/NHRk6ViiRp4mPXdRA9qahAlV/BDOK0Qfpg3F1iV8VBXS7xo6snTMqpGnSc/takDblHBJFT+F8wrRh2lDsV0JV6hCeqehI0vHtBp5jvTcqAa0ZQkTqvgpnFHouz5MGIrtSrhCFdI5DR1WOmbVyBOk50Y1oI1LuKSKn8J5hejDtKHYooTrVCFd0NAxpWOJGnmc9NyuBrRxCROq+E04rxB9mDYUm5NwhSqkSQ0dUzpm1cjjpOd2NaCNS5hQxW/CRYXow7Sh2JCE61QhzWnogNKxRI08QnpuVAPai4RLqngrnFfouz5MG4qtSLhCFdIyDR1QOmbVyH2l5y5qQHuRMKGKt8J5hejDtKFYv4SrVSEt1tDRpGNWjdxXem5XA9qXhEuqOCOcV4g+TBuKlUu4ThXSBzV0NOmYVSN3lJ5b1ID2KGFCFW+F8wp914dpQ7FaCVerQvq4hg4lHbNq5C7Sc7sa0E4lXFLFGeG8Qt/1YdpQrFPCdaqQrtXQoaRjVo3cLj03qgHtV8KEKs4I5xWiD7OGYoUSrlOFdIOGjiMds2rkRum5RQ1o7xImVHFeOK8QfZg2FGuTcJ0qpJs1dBzpmFYjt0jPjWpAe5cwrYozwnmFvuvDtKFYlYTrVCHdQ0MHkY5ZNXK19NyoBnQACROqOC+cV+i7PkwbijVIuFoV0v00dBDpmFYjV0jPjWpAh5EwrYozwkWF6MOsofh0CVerQrqrho4gHdNq5ArpuUUN6EgSplVxXrioEH2YNhSfLuFqVUj31tARpGNajXxUeq5WAzqYhGlVXBQuKkQfpg3FJ0q4WhXSYzS0e+mYViPLpecWNaDjSZhWxZRwXqHv+jBtKD5LwtWqkB6mod1Lx7QaWSg916kBHVjCtCouChcVog+zhuL5Eq5ThfR4De1bOqbVyBLpuVoN6MASplUxJVxUiD5MG4rnS7hOFdJTNLRv6ZhWI7PSc7Ua0LElTKviojClEH2YNhRPlnCdKqRnaWjH0jGtRiak52o1IEHCtCqmhIsKfdeHaUPxHAlXq0J6roZ2LB0TamRCeq5QA9IvEqZVMSVcVIg+zBqKJ0i4WhXS0zW0Y+mYUCNnpec6NSD9ImFaFTPCRYXow7SheLSEq1UhfZKG9iodE2rkvfRcpwak3yVMq2JGmFKIPkwbiodKuFoV0udpaK/ScUmNvJeeK9SAdE7ChCrmhYsKfdeHaUPxOAlXq0L6VA3tUjom1Miv0nOFGpAuSJhWxbxwUaHv+jBtKB4h4TpVSOvQ0C6l45Ia+Vd6PqoGpEkJ06qYF6YUog/ThuIREq5ThbQaDe1POi6pkb+l56NqQJqTMKuKeWFKIfowbSjuK+E6VUgr09D+pOOSGvlTej6qBqQFEqZVMS9MKfRdH6YNxR0lXKcKaX0a2pl0XFIjf0rPh9SAtEzCtCoWCVMKfdeHaUNxFwlXq0JapYZ2Jh1n1Uh6lqsB6YMSJlSxVJhS6Ls+TBiKu0i4ThXSijW0J+m4qGOhGpA+LmFaFUuFiwp914dpQ3GjhKtVIa1bQ3uSjvM6lqgB6SoJ06pYKkwp9F0fpg3FLRKuU4W0BQ3tSTrO6JhWA9INEmZVsVSYUui7PkwYiqslXKcKaTsa2o10nNcxoQak2yRMq+IDwpRC3/VhwlBcJ+EKVUhb09A+nGDoOKPjrBqQ7iFhVhVLhSmFvuvDtKH4qITrVCFtUEObduKMoeOHjvdqQLqfhGlVfECYUui7PkwYio9KuEIV0mY1tF0nLho66PhVDUj3ljCtio8JUwp914cJQ/EhCVeoQtqyhjbqxIyh5281ID1AwqwqPiZcVOiHPkwYiuUSPqoKafsa2qgTMxrSQyVMq+LDwkWFfujDJUOxUMJHVSHtRUMbdWJGQ3qchGlVfFiYUuiHPlwyFLMSrlCFtCMNbdGJRRrSIyRMq+IaYUqhH/pwyVBMS/ioKqTdaWijTsxoSI+QMKuKDwszCn3Xh0uGYlrCh1Qh7VRDG3ViRkO6u4RZVVwjTCn0Qx8uGYpLEj6qCmm/GtqoEzMa0t0lzKriw8KMQj/04ZKhOCvhQ6qQ9q6h7TpxUUO6u4QlqviwMKPQD304ayjOSliuCukYGtq0E2c0pEdImFXFNcKMQj/04ayheC9hoSqkI2loH07QkB4nYVYVVwpTCv3Uh7OG4lcJy1UhHUxDkuYkzKriemFKoZ/68N5Q/CphoSqkQ2pI0qSEJaq4UphR6Ic+nDUU/0pYogrpwBqSNClhVhXXC1MK/dSH94bibwlLVCEdXkOSLktYoorrhSmFfurDe0Pxp4QlqpAEDUm6LGFWFdcLUwr9pg9/Sk8N/GsoEmZVIekfDUm6IGFWFTcJUwr9dDrxXmskTKtC0u8aknROwhJVXC/MKPTD6cQlw8CEKiS905CkcxJmVXGTMKPQd6cT04aB96qQdEFDkt5JWKKKm4QZhb47nZg2DPyrCklzGpL0TsKsKm4VphT64XRi2jDwtyokLdCQpN8lzKriDsKUQt+dTizRGpIWa0jSLxJmVXEfYUqhH04nprWGpI9oSNI/EmZVcR9hSqGfTiemtYakj2hI0j8SZlVxH2FKoZ9OJ6a1hqSPaEjSXxJmVXEfYUah35xOXNIakj6oIUl/SZhVxX2EGYXeOp14rzUkfVxDkiBhVhV3E2YUuuh0ojUk3aAh6fASZlVxN2FGIUkP1ZB0bAlLVHE3YUYhSQ/VkHRsCbOquJswr5Ckh2pIOrCEJaq4mzCjkKRHa0g6sIRZVdxTmFFI0qM1JB1Vwqwq7izMKCTp0RqSDilhiSruLMwoJOnRGpKOJ2GJKu4vzCgk6dEako4nYYkq7izMKCTpCRqSDiZhiSruL8woJOkJGpIOJmFWFfcX5hWS9AQNSUeSMKuKhwgzCkl6joakw0hYooqHCDMKSXqOhqRjSFiiiocIMwpJepqGpGNIWKKKhwgzCkl6moakA0hYoopHCTMKSXqahqQDSJhVxaOEeYUkPU1D0t4lLFHFo4R5hSQ9TUPS3iXMquKBwrxCkp6mIWnXEpao4oHCjEKSnqkhab8SlqjiscKMQpKeqSFppxKWqOKxwrxCkp6pIWmnEpao4rHCjEKSnqwhaY8Slqji4cKMQpKerCFpdxKWqOLhwoxCkp6vIWl3EmZV8QxhRiFJz9eQtC8JS1TxDGFGIUnP15C0LwmzqniGMK+QpOdrSNqRhCWqeIYwr5Ck52tI2ouEJap4kjCvkKTna0jahYSFqniGMK+QpE/RkLQLCUtU8SRhXiFJn6IhafsSlqjiecKMQpI+S0PS9iXMquKpwoxCkj5LQ9LGJSxRxVOFGYUkfZaGpI1LmFXFU4UZhSR9ooakLUtYooqnCjMKSfpEDUmblbBEFc8WZhSS9IkakrYpYYkqni3MKCTpczUkbVDCElV8gjCjkKTP1ZC0QQlLVPEJwoxCkj5XQ9LWJCxRxScI8wpJ+lwNSZuSsFAVnyDMKyTpczUkbUrCElV8grBIIUmfqyFpOxIWquIThHmFJH26hqTtSFiiis8R5hWS9OkakjYiYYkqPk2YUUjSGjQkbUHCElV8pjCjkKQ1aEhavYSFqvhMYUYhSWvQkLR6CUtU8ZnCjEKSVqIhad0SFqriM4UZhSStREPSuiUsUcUnCzMKSVqJhqQVS1iiik8WZhSStB4NSSuWsEQVnyzMKCRpPRqS1iphiSo+X5hRSNJ6NCStUsISVXy+MK+QpPVoSFqfhCWqWIUwr5Ck9WhIWp+EWVWsQlikkKT1aEhamYQlqliFMK+QpFVpSFqZhFlVrEWYV0jSqjQkrUnCElWsRZhRSNLaNCStRsISVaxImFFI0to0JK1DwkJVrEiYUUjS2jQkrUPCElWsSJhRSNIKNSStQMISVaxLmFFI0go1JK1AwqwqVifMKCRphRqSPlvCElWsS5hRSNI6NSR9toRZVaxOmFFI0jo1JH2qhCWqWJ0wo5CkdWpI+jwJS1SxOmFeIUnr1JD0SRKWqGKNwrxCktapIemTJCxRxeqERQpJWqeGpM+QsEQVaxTmFZK0Wg1JT5ewRBUrFeYVkrRaDUlPlzCrivUKMwpJWrOGpOdKWKKK9QozCklas4ak50qYVcWqhRmFJK1ZQ9ITJSxRxXqFGYUkrVxD0rMkLFHFqoUZhSStXEPSsyQsUcWqhRmFJK1cQ9JTJCxRxaqFGYUkrV9D0lMkzKpi7cKMQpLWryHp8RJmVbEBYUYhSevXkPRgCUtUsXZhXiFJ69eQ9EgJS1SxAWFeIUnr15D0SAlLVLF2YV4hSZvQkPQwCUtUsQFhXiFJm9CQ9DAJs6rYhjCvkKRNaEh6jIRZVWxGmFFI0lY0JD1AwhJVbEaYUUjSVjQkPUDCrCq2JMwoJGkrGpLuLWGJKjYjzCgkaUMaku4tYVYVWxJmFJK0IQ1Jd5Uwq4otCfMKSdqQhqT7SZhVxcaEGYUkbUtD0v0kzKpiY8KMQpK2pSHpThJmVbExYV4hSdvSkHQnCbOq2Jgwr5CkbWlIuoeEWVVsT5hXSNK2NCTdLGFWFZsUZhSStDkNSbdJmFXFVoUZhSRtTkPSDRKWqGKrwoxCkjanIekGCbOq2Kowo5CkLWpIulbCElVsVZhRSNIWNSRdK2FWFVsV5hWStEUNSVdJmFXFhoUZhSRtVEPSxyUsUcWGhRmFJG1UQ9LHJcyqYsPCjEKStqsh6YMSlqhiw8KMQpK2qyHpgxJmVbFtYUYhSdvVkPQRCbOq2Lwwo5Ck7WpIWixhVhV7EGYUkrRdDUnLJCxRxeaFeYUkbVdD0jIJs6rYgzCjkKRNa0haJmFWFXsQZhSStGkNSQskzKpiD8KMQpK2riFpgYRpVexEmFFI0tY1JM1JmFbFfoQZhSRtXUPSpIRpVexKmFJI0g40JF2WMKuKXQlTCknagYakyxKmVbErYUohSfvQkHRBwrQq9iZMKSRpHxqSzkmYVcXehCmFJO1DQ9I5CdOq2Jswo5CkfWhIeidhVhV7E2YUkrQPDUnvJEyrYofCjEKS9qEh6XcJ06rYpzClkKTdaEj6XcKEKnYrTCkkaTcakn6RMK2KfQpTCknak4akfyRMq2K3wpRCkvakIekvCbOq2Kcwo5CkPWlI+kvCtCp2K0wppH07nWgNHUpDEiRMq2LPwpRC2qXTifdaQ0fQkAQJE6rYszCjkPbndOKS1tDuNaTDS5hWxZ6FGYW0M6cT01pD+9aQji1hWhU7F2YU0s6cTkxrDe1bQzqwhGlV7F+YUkh70oU//bdnWmto3xrSgSVMq2LnwoxC2rQuvPHfniVaQzvWkI4qYVoV+xemFNIWdWHaf3umtYb2rSEdVcK0KvYvTCmkrejCcv/tmdYa2reGdEgJ06o4hHBRIa1cF64wFqcT01pD+9aQDilhWhWHEC4qpBXqwnXG4lenE5e0hnavIR1PwrQqDiFMKaQ16MKNxuKs04n3WkNH0JAOJmFaFUcRphTSJ+rC7cbikj78Kz2toUNpSEeSMK2KAwlTCun5unCjsViiD78aCh1KQzqShAlVHEiYUkjP1IXbjcVCffjVUOhoGtJhJEyr4kDClEJ6nC7c11h8SB9+NRQ6moZ0GAkTqjiWMKWQ7q4LdzQW1+nDr4ZCB9SQjiFhQhWHEy4qpHvpwn2NxY368Kuh0AE1pGNIuKSKwwlTCul2Xbi7sbhRH94YCh1QQzqAhAlVHE6YUkjX6cIjjMVd9OG9odABNaQDSLikiiMKUwppuS48yFjcVx/eGAodU0Pau4QJVRxOmFJIs7rwIGPxIH14byh0TA1p7xIuqeKIwpRCmtCFRxiLR+vDG0Ohw2pIu5YwoYojChcV0ntdeJyxeII+vDcUOqyGtF8JE6o4qHBRIf2pC482Fs/UhzeGQkfWkHYqYUIVBxWmFDq4LjzOWHyKPrw3FDqyhrRHCROqOK5wUaFj6sKjjcVn6cNZQ6Eja0h7lHBJFYcWLip0NF14qLH4dH14byh0cA1pdxImVHFo4aJCu9eF5xiLlejDe0Ohg2tIu5NwSRWHFqYU2rEuPNpYrE0fzhoKHVxD2peES6o4unBRoZ3pwtOMxTr14b2hkBrSjiRMqOLowkWFdqMLTzAWK9eHs4ZCakg7knBJFSKcV2jTuvA0Y7EJfbhkKKSGtCMJl1RxdOGiQhvVhScYi83pw1lDIf2pIe1FwiVViHBRoa3owjONxUb14ayhkP7UkPYi4awq9F24qND6deE5xmLr+nDWUEh/a0i7kHBWFfohnFdonbrwZGOxA324ZCikvzWkXUh4rwr9FM4rtDZdeI6x2Jk+XDIU0r8a0vYlvFeFfgrnFVqPLjzNWOxSHy4ZCqXHgDQAABEqSURBVOlfDWnjEs6qQj+F8wp9ri482VjsWB/OGgrpVw1pyxLOqkK/CecV+hRdeJqxOIg+XDIU0q8a0pYlvFeF3grnFXqaLjzZWBxKHy4ZCulXDWmzEt6r4oBO0LgsnFfoCbrwfGNxNH24ZCikNxrSZiW8V8VxnDij8U44r9BDdeFpxuLg+nDJUEhvNKTNSnivioM4cVHjd+G8QvfVhecbC/VhwlBIbzSkbUp4r4qDODGj8Y9wUaF76cLTjIXe6MMlQyG915A2KOG9KvYkPRP6gWmNf4TzCt2uC08zFjqrDxOGQnqvIW1QwhtVbEt6btEPTGv8I5xX6DpdeLKx0CV9mDAU0lkNaWsS3qtitdJzX/3AEo2/hDMKfVQXnmYstFAfJgyFdFZD2pSE96pYifQ8Rz8wrfGXcF6h5brwNGOh5fowbSiksxrSpiS8UcVnSc9n6QemNf4Szis0oQvPNxb6qD5MGArpkoa0KQm/quJp0rMe/cC0xl/CeYXO6sLTjIVu1IdLhkKa0JC2I+GNKh4nPStUA387cVHjH+GMQv/qwpONhe6iDxOGQprQkDYi4Y0q7is9q1UD7504o/GPcF6hP3XhycZCd9SHCUMhTWhIW5DwRhW3S8+a1cC0dPytHxk6auQ34bzi4LrwHGOhB+nDhKGQpjWkLUh4o4orpGflamChdLxRI78JZxQH1IUnGws9Th+mDYU0rSFtQcK/qvio9KxZDVwhHW/UyG/CGcWhdOE5xkJP04cJQyHNakirl/CrKpZIz8rVwC3S8asaeSucURxEF55jLPRMfZg2FNKshrRuCb+qYkJ61qwG7iUdb9TIb8IZxY514ZnGQp+iDxOGQlqiIa3b6YXhP/ytivfSs1o18CDpeKNGfhPOKHapC88xFvp0fZgwFNISDWmVTi+8177xr/SsUA08QTreq5HfhDOK3ejC04yFVqIP04ZCWqIhrc/phUuGL6xKDTxfOt6rkZ/CGcU+dOEJxkIr1IcJQyEt1JBW5vTCtOELn6UG1iAdb9TIb8IZxUZ14WnGQqvVh2lDIS3UkFbm9MK04QvPUQMrlI73auQ34Yxic7rwBGOhTejDhKGQlmtIK3N6Ydrwhburga1Ix3s18pvwVrEVXXiOsdCG9GHaUEjLNaQ1Ob2wxPCFq9XAdqXjrBr5KZxRrF8XHm0stEV9mDUU0nINaWVOL0wbvjChBnYsHe/VyG/CGcU6deE5xkLb1YdpQyF9SENamdML09o3jikdZ9XIb8JbxQp14aHGQvvQh1lDIX1IQ1qZ0wvT2jeOKR1n1chvwlvFenTh0cZCe9KHaUMhfVRDWp/TC5e0bxxTOi6pkd+Et4o16MLjjIV2qQ+zhkL6qIa0SqcX3mvfOKx0nFUjb4XfFJ+oCw81Ftq3PkwbCukKDWndTi+0bygdZ9XIb8JbxZN14aHGQsfRh2lDIV2hIWn10nFJjfwm/KZ4si48wljogPowbSik6zQkrV46LqmR34TfFE/QhccZCx1WH6YNhXSdhqTVS8dZNfJW+E3xOF14kLGQ+jBrKKTrNCStWzouqZHfhN8Uj9CFRxgL6Vd9mDYU0tUaktYtHZfUyG/Cb4o76sLdjYV0Vh9mDYV0tYakFUvHhBr5TfipuJcu3N1YSBP6MG0opFs0JK1YOi6pkbfCT8XtunBHYyEt0YdZQyHdoiFpxdJxSY28FX4qrtOFOxoL6aP6MG0opBs1JK1VOibUyG/CT8VHdeEuxkK6RR9mDYV0o4aktUrHhBr5TfipWK4LtxsL6S76MG0opNs1JK1SOibUyFvha8/rAMWsLtxuLKT76sOsoZBu15C0SumYUCP/+nrivdfGWV240VhID9KHaUMh3UVD0iqlY0KN/O3riUteG7/qwtXGQnq0PswaCukuGpLWJx0TauRvX09Me2104QpjIT1TH5YYCukuGpLWJx0TauRvX09M+9+BDxkL6VP0YYmhkO6iIWl90jGhRv729cS0/x2YNhbSp+vDEkMh3UtD0sqkY0KN/O3riSX+d+CNsZBWpQ+zhkK6o4aklUnHhBr519cT0/534E9jIa1ZH2YNhXRHDUkrk44JNfKvryemvTak9evDtKGQ7qshaU3SMa1G/vX1xLTXhrRyfZg1FNJ9NSStSTom1MgbX09c8tqQ1q8P04ZCuruGpDVJx4Qaee/rifdeG9L69WHWUEh315C0GumYViMTvp54bUgb0odpQyE9QkPSaqRjQo1Ie9KHWUMhPUJD0jqkY1qNSHvSh2lDIT1IQ9I6pGNajUi70YdZQyE9SEPSOqRjWo1Iu9GHaUMhPU5D0jqkY0KNSLvRh1lDIT1OQ9IKpGNajUi70YdZQyE9TkPSCqRjWo1I+9CHJYZCepyGpM+Wjlk1Iu1DH2YNhfRQDUmfLR3TakTahz4sMRTSQzUkfbZ0TKsRaR/6MGsopEdrSPpU6ZhVI9IO9GGJoZAerSHpU6VjWo1I+9CHWUMhPUFD0udJx6wakXagD0sMhfQEDUmfJx2zakTagT7MGgrpORqSPk86ptWItAN9WGIopOdoSPo86ZhWI9IO9GHWUEhP05D0SdIxq0akrevDEkMhPU1D0idJx7QakXagD7OGQnqmhqRPko5pNSJtXR+WGArpmRrS/98e3NjGEbNnAHzeElLL1mKXQNZElmDXsrWkBOVDAgSGJe3Jtn6Ox5nhK4wjN/UzsLo2ctPsgU9WAb7COHJTPwNLayNvMXvgk1WArzCOXOtnYHVt5KbZA5+vAny6ceSmfgaW1kbeYvbA56sAn24cuamfgaW1kZtmD3yJCvDpxpFr/QwsrY28xeyBL1EBPtc4clM/A0trIzfNHvgqFeBzjSPX+hlYXRu5afbAV6kAn2scudbPwOrayLXZA1+oAnyiceSmfgaW1kZumj3whSrAJxpHbupnYGlt5KbZA1+oAnyiceRaPwNLayNvMXvgC1WAzzKO3NTPwNLayE2zB75WBfgs48hN/Qysq428xeyBr1UBPss4cq2fgaW1kZtmD3y5CvBZxpFr/Qysq428xeyBL1cBPsU4clM/A+tqIzfNHrgHFeBTjCPX+hlYVxt5i9kD96ACfLxx5KZ+BtbVRm6aPXAnKsDHG0du6mdgUW3kLWYP3IkK8PHGkWv9DKyrjdw0e+B+VIAPNo7c1M/AotrIW8weuB8V4IONIzf1M7CoNnLT7IG7UgE+2DhyrZ+BRbWRt5g9cFcqwAcbR671M7CoNnLT7IF7UwE+0jhyUz8DK2ojbzF74N5UgI80jlzrZ2BRbeSm2QN3qAJ8pHHkWj8DK2ojbzF74A5VgA8zjtzUz8CK2shNswfuUwX4MOPITf0MLKeNvMXsgftUAT7MOHKtn4EVtZGbZg/crQrwMcaRm/oZWFEbuWn2wN2qAB9jHLnWz8Ci2si12QP3rAJ8jHHkWj8DK2ojN80euGcV4AOMIzf1M7CiNnJt9sCdqwAfYBy51s/AitrITbMH7lwF+ADjyLV+BlbURq7NHrh/FeADjCMX+hlYURu5afbA/asA720cudbPwIrayLXZA0uoAO9tHLnWz8By2shNsweWUAHe2zhyrZ+B5bSRa7MHVlEB3ts4cqGfgeW0kZtmD6yiAryrceRaPwPLaSM3zR5YRQV4V+PIhX4GltNG3mL2wCoqwLsaRy70M7CWNvIWswcWUgHezzhyrZ+BtbSRt5g9sJAK8H7GkQv9DCynjdw0e2AtFeD9jCMX+hlYThu5NntgORXg/YwjF/oZWEsbuWn2wHIqwDsZRy70M7CcNnJt9sCKKsA7GUcu9DOwljZy0+yBFVWAdzKOXOhnYC1t5NrsgUVVgPcwjlzoZ2AtbeSm2QOLqgDvYRy50M/AWtrItdkD66oA72EcudDPwELayE2zB9ZVAf7ZOHKhn4G1tJFrsweWVgH+2ThyoZ+BhbSRm2YPLK0C/LNx5DX9DKyljVybPbC6CvDPxpHX9DOwljZybfbA6irAvxlHLvQzsJA2cm32wAOoAP9mHHlNPwNraSPXZg88gArwb8aR1/QzsJA2ctPsgQdQAf7NOPKifgbW0kauzR54DBXgH4wjr+lnYCFt5KbZA4+hAvyDceQ1/QwspI1cmz3wMCrAPxhHXtTPwELayE2zBx5GBfgH48iL+hlYSBu5NnvgkVSAvzWOvKafgYW0kQuzBx5MBfhb48iL+hlYSBu5NnvgwVSAvzWOvKifgYW0kQuzBx5PBfhb48hz/QwspI1cmz3weCrAXxlHXtTPwELayIXZAw+pAvyVceS5fgbW0kYuzB54SBXgr4wjz/UzsJA2cm32wEOqAH9lHPlNPwNraSMXZg88qgrw58aR5/oZWEgbuTZ74FFVgD83jjzXz8BC2siF2QMPrAL8uXHkN/0MLKSNXJg98NgqwJ8bR37Tz8BC2siF2QOPrQL8uXHkV/0MrKWNvGb2wMOrAH9oHPlNPwMLaSMXZg88vArwh8aR3/QzsIo2cm32wMOrAH9oHGnnt3n8zP/qZ2AhbeTC7IEdVIA3e8q3PFP5GVhHG7kwe2AHFeBtnvItr6j8DKygjVyYPbCJCvAGT/mWS5WfgbvXRl4ze2AfFeANnvItlyo/A/etjVyYPbCPCvAGT/mWS5WfgfvWRl4ze2ArFeCWp3zLG1R+Bu5YG3nN7IGtVIA3eMq3XKr8DNyxNvKa2QO7qQBv8JRvuVT5GbhjbeQ1swd2UwHe4CnfcqnyM3DH2siLZg9sqAK8zVO+5RWVn4E71kZeM3tgQxXgzZ7yLc9UfgbuWxt5zeyBDVWAP/eUb5WfgRW0kdfMHthTBYCH1kZeNHtgWxUAHlcbedHsgZ1VAHhcbeRFswd2VgHgQbWR18we2FkFgAfVRl40e2BzFQAeVBt5bvYAFQAeURt50ewBKgA8ojby3OwB/qMCwCNqI8/NHuA/KgA8nDbyotkD/EcFgIfTRp6bPcD/qQDwcNrIb2YP8P8qADyWNvLc7AH+XwWAx9JGfjN7gF9VAHggbeS52QP8qgLAA2kjv5k9wG8qADyKNvLc7AF+UwHgUbSR38we4LkKAI+ijfxq9gAvqgDwKNrIr2YP8KIKAA+hjfxq9gCvqQDwENrIr2YP8JoKAOtrI7+aPcCFCgDrayO/mj3AhQoA62sj47/S/zv/MXuAaxUAVvb0I8/V9wAXKgAs6+lHXlPfA7ymAsCann7kWn0P8KIKAGt6+pFr9T3AiyoArOnpR67V9wAvqgCwoKcfeYv6HuC5CgBrevqRa/U9wIsqAKzp6Ueu1fcAL6oAsKanH7lW3wO8qALAsp5+5DX1PcBrKgCs7OlHnqvvAS5UAHgITz9S3wO8RQUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAzFQCAzVQAADZTAQDYTAUAYDMVAIDNVAAANlMBANhMBQBgMxUAgM1UAAA2UwEA2EwFAGAz/wOa8Z6mBCsXAwAAAABJRU5ErkJggg==',
}

print('그림 모델을 받는 중입니다. 처음 한 번만 2분쯤 걸립니다...')
cn = ControlNetModel.from_pretrained('lllyasviel/control_v11p_sd15_openpose',
                                     torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V5.1_noVAE', controlnet=cn,
    torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('모델 준비 끝. 이제 그립니다.')
print('')

os.makedirs('결과', exist_ok=True)
made = []
for i, name in enumerate(sorted(POSES.keys()), 1):
    자세 = Image.open(io.BytesIO(base64.b64decode(POSES[name]))).convert('RGB')
    g = torch.Generator('cuda').manual_seed(같은사람_번호)
    img = pipe(설명, image=자세, negative_prompt=빼고싶은것,
               num_inference_steps=28, guidance_scale=7.0,
               controlnet_conditioning_scale=1.0, generator=g).images[0]
    path = os.path.join('결과', name + '.png')
    img.save(path)
    made.append(path)
    print(str(i) + '/' + str(len(POSES)) + '  ' + name)
    나란히 = np.hstack([np.array(자세.resize((384, 384))),
                      np.array(img.resize((384, 384)))])
    display(Image.fromarray(나란히))

with zipfile.ZipFile('결과.zip', 'w') as z:
    for p in made:
        z.write(p)

print('')
print('끝났습니다. 왼쪽이 우리가 넣은 자세, 오른쪽이 AI가 그린 사람입니다.')
try:
    from google.colab import files
    files.download('결과.zip')
except Exception:
    print('자동 내려받기가 안 되면 왼쪽 폴더 아이콘에서 결과.zip 을 받으세요.')